# Graph RAG — COSORA (experimental)

Pipeline COSORA + Knowledge Graph generado con **kg-gen** sobre los chunks ya indexados en ChromaDB.

**Idea:** dense (E5) + BM25 + **triples del grafo** fusionados con RRF.
Genera el grafo una sola vez (offline) con `gpt-4o-mini`, lo persiste en JSON
y opcionalmente lo sube a GCS junto al Chroma.

**Prerequisito:** haber ejecutado `notebooks/ingestion_pipeline.ipynb` con `EMBED_BACKEND="e5"`.

> Autocontenido. Funciona en local (kernel `.venv`) y en Colab.


## 0. Setup
Instalación de dependencias, auto-detección Colab/local y código COSORA embebido.
Incluye `kg-gen` (PyPI, no `git clone`).


In [1]:
%pip install -q chromadb sentence-transformers rank_bm25 transformers accelerate python-dotenv python-docx nltk pandas torch kg-gen google-cloud-storage

# antiword (para .doc legacy) — sólo lo necesita KG_SOURCE="original"
import shutil, subprocess
if shutil.which("antiword") is None:
    try:
        subprocess.run(["apt-get", "install", "-y", "-q", "antiword"], check=False)
    except Exception:
        print("⚠️  antiword no se pudo instalar. KG_SOURCE='original' sólo leerá .docx.")

import nltk
nltk.download('punkt', quiet=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 143.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 95.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 336.6 kB/s eta 0:00:00a 0:00:01
   ━━━━━━

True

In [24]:
# ═══════════════════════════════════════════════════════════════════════
# AUTO-DETECT: Colab vs Local  +  Configuración
# ═══════════════════════════════════════════════════════════════════════
import glob
import json
import os
import re
import subprocess
import sys
import time
import unicodedata
from pathlib import Path
from typing import Any

import chromadb
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from transformers import AutoModel, AutoTokenizer

IN_COLAB = "google.colab" in sys.modules

# ─── Configuración ────────────────────────────────────────────────────
RUNTIME = "auto"            # "auto" | "colab" | "local"
EMBED_BACKEND = "e5"        # fijado a e5 para alinear con producción (src/app.py)

GEN_BACKEND = "openai"

RETRIEVAL_K = 100
TOP_N = 10
RRF_K = 60
RRF_MIN_SCORE = 0.015
SCORE_RATIO = 0.60

# ─── Graph RAG ────────────────────────────────────────────────────────
KG_MODEL = "openai/gpt-4o-mini"   # litellm-style, lo usa kg-gen
KG_CONTEXT = "Actas de obra ferroviaria en España. Personas, empresas (UTE, DF, DEO), elementos constructivos, incidencias, fechas."
KG_SOURCES = ["chunks", "hybrid", "original"]   # se generan y comparan los 3
                                  #   chunks   = N chunks tal cual salen de Chroma (más granular, menos contexto)
                                  #   hybrid   = chunks AGRUPADOS por doc_id → un texto por acta (sin tocar .doc/.docx)
                                  #   original = lee .doc/.docx desde RAW_DOCS_PATH (replica ingest.py)
KG_SOURCE_PRIMARY = "hybrid"      # cuál se usa para las pruebas interactivas (secciones 3-7)
SUBSET_SIZE = 10                  # nº de elementos para pruebas (chunks o actas según fuente). None = todos
KG_CHUNK_SIZE = 5000              # tamaño de chunk interno de kg-gen (chars)
KG_CLUSTER = True                 # entity resolution por defecto
KG_CLUSTER_BY_SOURCE = {          # override por fuente (None = usar KG_CLUSTER)
    "chunks": True,
    "hybrid": True,
    "original": False,            # en original el clustering tarda 30+ min y aporta poco (ver análisis)
}
GRAPH_TOP_K = 5                   # nº de triples a recuperar por query
GRAPH_MIN_COS = 0.80              # umbral mínimo de coseno para que un triple boostee chunks
                                  # (subido de 0.72 → 0.80 tras run sub10: con 100+ triples
                                  #  siempre había alguno >0.72 metiendo ruido)
BALANCED_SAMPLING = True          # muestreo equilibrado entre familias de doc_id (ver prepare_inputs)
FORCE_REGEN = []                  # lista de fuentes a regenerar aunque ya exista el JSON.

# ─── Eval avanzado ────────────────────────────────────────────────────
EVAL_K_LIST = [5, 10]             # K's para Precision@K / NDCG@K (más robusto que un solo K)
CROSS_JUDGE_MODEL = "gpt-4o"          # "gpt-4o" para validar con un juez distinto (None = solo gpt-4o-mini)
                                  # ⚠️ multiplica el coste ×N por las llamadas extra
ANSWER_EVAL_ENABLED = True        # eval de calidad de respuesta (correctness/faithfulness/citation)
MATCH_STRATEGY = "wordset"        # "substring" (legacy) | "wordset" (D1: palabras completas con solape)
MATCH_MIN_OVERLAP = 0.5           # para "wordset": fracción mínima de palabras del triple en el chunk

# ─── M3: Entity linking ───────────────────────────────────────────────
ENTITY_MIN_COS = 0.85             # umbral query-entidad ↔ entidad-grafo (más estricto que GRAPH_MIN_COS)
ENTITY_SOURCE = "original"        # grafo para entity linking ("chunks" | "hybrid" | "original")
ENTITY_EXTRACT_MODEL = "gpt-4o-mini"
RAW_DOCS_PATH_OVERRIDE = None     # None = autodetectado (data/raw o Drive). Solo si KG_SOURCE="original"

# Persistencia
GRAPH_JSON_NAME = "graph_actas_e5.json"
GCS_BUCKET = os.getenv("GCP_BUCKET_NAME", "")
GCS_GRAPH_PREFIX = "graph"

if RUNTIME == "auto":
    RUNTIME = "colab" if IN_COLAB else "local"

if RUNTIME == "colab" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# ─── .env ─────────────────────────────────────────────────────────────
from dotenv import load_dotenv
if RUNTIME == "colab":
    load_dotenv("/content/drive/MyDrive/variablentorno/.env")
elif Path(".env").exists():
    load_dotenv(".env")
elif Path("../.env").exists():
    load_dotenv("../.env")
elif Path("../../.env").exists():
    load_dotenv("../../.env")


# ─── Rutas ────────────────────────────────────────────────────────────
def resolve_paths(runtime, chroma_path_override=None):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        chroma_path = chroma_path_override or f"{docs_dir}/chroma_db"
        graph_dir = f"{docs_dir}/graph"
    else:
        nb_dir = Path(".").resolve()
        if nb_dir.name in ("experiments", "notebooks"):
            root = nb_dir.parent if nb_dir.name == "experiments" else nb_dir.parent
            root = nb_dir.parents[1] if nb_dir.name == "experiments" else nb_dir.parent
        else:
            root = nb_dir
        docs_dir = str(root / "data" / "raw")
        chroma_path = chroma_path_override or str(root / "data" / "chroma_db")
        graph_dir = str(root / "data" / "graph")
    Path(graph_dir).mkdir(parents=True, exist_ok=True)
    return docs_dir, chroma_path, graph_dir

# ═══════════════════════════════════════════════════════════════════════
# COSORA SHARED — código embebido
# ═══════════════════════════════════════════════════════════════════════

EMBED_BACKENDS: dict[str, dict[str, Any]] = {
    "mrbert": {
        "model_id": "BSC-LT/MrBERT-es",
        "collection": "cosora_chunks_mrbert",
        "type": "transformers_cls",
        "query_prefix": "",
        "doc_prefix": "",
    },
    "e5": {
        "model_id": "intfloat/multilingual-e5-base",
        "collection": "cosora_chunks_e5",
        "type": "sentence_transformers",
        "query_prefix": "query: ",
        "doc_prefix": "passage: ",
    },
}

TECH_TOKEN_PATTERN = re.compile(
    r"^(AR-\d+|UTE|DF|DEO|DO|PPI|[A-Z]{2,}\d*|\d+[A-Z-]+)$",
    re.IGNORECASE,
)

STOPWORDS_ES = {
    "de","la","que","el","en","y","a","los","del","se","las","por","un",
    "para","con","no","una","su","al","es","lo","como","más","pero","sus",
    "le","ya","o","este","sí","porque","esta","entre","cuando","muy","sin",
    "sobre","también","me","hasta","hay","donde","quien","desde","todo","nos",
    "durante","todos","uno","les","ni","contra","otros","ese","eso","ante",
}


def backend_config(backend):
    if backend not in EMBED_BACKENDS:
        raise ValueError(f"EMBED_BACKEND desconocido: {backend}")
    return EMBED_BACKENDS[backend]


class Embedder:
    def __init__(self, backend):
        self.backend = backend
        self.cfg = backend_config(backend)
        self._st_model = None
        self._model = None
        self._tokenizer = None

    def _load(self):
        if self.cfg["type"] == "sentence_transformers":
            if self._st_model is None:
                from sentence_transformers import SentenceTransformer
                self._st_model = SentenceTransformer(self.cfg["model_id"])
            return
        if self._model is None:
            self._tokenizer = AutoTokenizer.from_pretrained(self.cfg["model_id"])
            self._model = AutoModel.from_pretrained(self.cfg["model_id"])
            self._model.eval()

    def embed_one(self, text, *, is_query=False):
        self._load()
        if self.cfg["type"] == "sentence_transformers":
            prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
            vec = self._st_model.encode(prefix + text)
            return vec.tolist() if hasattr(vec, "tolist") else list(vec)
        assert self._tokenizer is not None and self._model is not None
        inputs = self._tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = self._model(**inputs)
        return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

    def embed_batch(self, texts, *, is_query=False, batch_size=16):
        self._load()
        if self.cfg["type"] == "sentence_transformers":
            prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
            prefixed = [prefix + t for t in texts]
            vecs = self._st_model.encode(prefixed, batch_size=batch_size, show_progress_bar=True)
            return vecs.tolist()
        out = []
        for t in texts:
            out.append(self.embed_one(t, is_query=is_query))
        return out


def strip_doc_prefix(text, backend):
    prefix = backend_config(backend).get("doc_prefix", "")
    if prefix and text.startswith(prefix):
        return text[len(prefix):]
    return text


def get_chroma_collection(chroma_path, backend):
    cfg = backend_config(backend)
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_collection(cfg["collection"]), cfg


def tokenize_bm25(text, stemmer=None):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    words = [w for w in text.split() if w and w not in STOPWORDS_ES]
    if stemmer is None:
        return words
    out = []
    for w in words:
        if TECH_TOKEN_PATTERN.match(w):
            out.append(w)
        else:
            out.append(stemmer.stem(w))
    return out


def build_bm25_index(collection):
    try:
        from nltk.stem.snowball import SnowballStemmer
        stemmer = SnowballStemmer("spanish")
    except Exception:
        stemmer = None
    data = collection.get()
    docs = data["documents"]
    metas = data["metadatas"]
    tokenized = [tokenize_bm25(d, stemmer) for d in docs]
    return BM25Okapi(tokenized), docs, metas, stemmer


def dense_search(collection, embedder, query, k=100):
    q_vec = embedder.embed_one(query, is_query=True)
    res = collection.query(query_embeddings=[q_vec], n_results=k)
    return [
        {"text": doc, "meta": meta, "rank_dense": i}
        for i, (doc, meta) in enumerate(zip(res["documents"][0], res["metadatas"][0]))
    ]


def bm25_search(query, bm25_index, docs, metas, stemmer, k=100):
    scores = bm25_index.get_scores(tokenize_bm25(query, stemmer))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {"text": docs[i], "meta": metas[i], "rank_bm25": rank}
        for rank, i in enumerate(top_idx)
    ]


def rrf_fusion_baseline(dense, bm25, rrf_k=60, top_n=10):
    """RRF clásico: dense + bm25 (baseline sin grafo)."""
    scores = {}
    for item in dense:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_dense"])
    for item in bm25:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_bm25"])
    return sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_n]


# ─── Inicializar ──────────────────────────────────────────────────────
DOCS_DIR, CHROMA_PATH, GRAPH_DIR = resolve_paths(RUNTIME)
collection, cfg = get_chroma_collection(CHROMA_PATH, EMBED_BACKEND)
embedder = Embedder(EMBED_BACKEND)
bm25_index, all_docs, all_metas, stemmer = build_bm25_index(collection)

print(f"IN_COLAB={IN_COLAB}  RUNTIME={RUNTIME}")
print(f"EMBED_BACKEND={EMBED_BACKEND}")
print(f"Chroma: {cfg['collection']} ({collection.count()} chunks) @ {CHROMA_PATH}")
print(f"Graph dir: {GRAPH_DIR}")
print("\n✅ Setup COSORA listo")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  RUNTIME=colab
EMBED_BACKEND=e5
Chroma: cosora_chunks_e5 (582 chunks) @ /content/drive/MyDrive/RAG_UPC_Final_project/chroma_db
Graph dir: /content/drive/MyDrive/RAG_UPC_Final_project/graph

✅ Setup COSORA listo


## 1. Retrieval híbrido baseline (dense + BM25)
Idéntico al de `src/app.py` y `query_pipeline.ipynb`. Lo usaremos como **baseline** contra el cual comparar Graph RAG.


In [3]:
def retrieve_baseline(query, top_n=TOP_N):
    d = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    b = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    return rrf_fusion_baseline(d, b, rrf_k=RRF_K, top_n=top_n)


# Sanity check
preview = retrieve_baseline("¿Qué se decidió sobre el talud?", top_n=3)
for h in preview:
    print(f"[{h['meta']['doc_id']}] score={h['score']:.4f} | {strip_doc_prefix(h['text'], EMBED_BACKEND)[:90]}...")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

[254275-DO-AVO-15-V07] score=0.0333 | adas en los informes de revisión del PC y PGMA. AR-29 Estabilidad del talud || Posterior a...
[254275-DO-AVO-14-V07] score=0.0325 | envía en resultado de la verificación de la documentación aportada por la UTE sobre la est...
[254275-DO-AVO-16-V07] score=0.0323 | iará el certificado a la DF. AR-21 Plano camino provisional || DF solicita a la UTE que ap...


## 2. Construcción del grafo con `kg-gen`

Generamos triples `(sujeto, relación, objeto)` desde los chunks de Chroma usando `gpt-4o-mini` vía `kg-gen`.

**Estrategia para no quemar API:**
- `SUBSET_SIZE` controla cuántos chunks procesamos (configurado arriba en Setup).
- `cluster=True` ejecuta entity resolution (UTE = contratista = constructora, etc.).
- El grafo se persiste en JSON; si ya existe, se carga en vez de regenerarse.

**Coste estimado:** ~0,03 USD por los ~400 chunks completos con `gpt-4o-mini`.


In [4]:
# Preparar textos para kg-gen según KG_SOURCE
from collections import defaultdict

def _prepare_from_chunks():
    data = collection.get(include=["documents", "metadatas"])
    texts = [strip_doc_prefix(d, EMBED_BACKEND) for d in data["documents"]]
    sources = [m["chunk_id"] for m in data["metadatas"]]
    return texts, sources, "chunks de Chroma (tal cual)"


def _prepare_hybrid():
    data = collection.get(include=["documents", "metadatas"])
    by_doc = defaultdict(list)
    for d, m in zip(data["documents"], data["metadatas"]):
        by_doc[m["doc_id"]].append(strip_doc_prefix(d, EMBED_BACKEND))
    doc_ids = sorted(by_doc.keys())
    texts = ["\n\n".join(by_doc[d]) for d in doc_ids]
    return texts, doc_ids, "actas reconstruidas desde Chroma (chunks agrupados por doc_id)"


def _prepare_original():
    raw_dir = RAW_DOCS_PATH_OVERRIDE
    if raw_dir is None:
        raw_dir = DOCS_DIR if Path(DOCS_DIR).exists() else None
        if raw_dir is None:
            candidates = [
                Path("../data/raw"),
                Path("../../data/raw"),
                Path("data/raw"),
            ]
            raw_dir = next((str(c.resolve()) for c in candidates if c.exists()), None)
    if raw_dir is None or not Path(raw_dir).exists():
        raise FileNotFoundError(
            f"No se encontró carpeta de actas originales. Define RAW_DOCS_PATH_OVERRIDE."
        )

    # Loaders idénticos a src/ingest.py
    from docx import Document
    import subprocess as _sp

    def _normalize(t):
        return re.sub(r"\s+", " ", t).strip()

    def _extract_docx(fp):
        try:
            doc = Document(fp)
        except Exception as e:
            print(f"  ⚠️ {fp.name}: {e}")
            return None
        parts = []
        for p in doc.paragraphs:
            t = _normalize(p.text)
            if t:
                parts.append(t)
        for table in doc.tables:
            for row in table.rows:
                row_texts, seen = [], set()
                for cell in row.cells:
                    t = _normalize(cell.text)
                    if t and t not in seen:
                        seen.add(t)
                        row_texts.append(t)
                if row_texts:
                    parts.append(" || ".join(row_texts))
        return "\n".join(parts).strip() or None

    def _extract_doc(fp):
        try:
            r = _sp.run(["antiword", str(fp)], capture_output=True, text=True)
            return r.stdout.strip() if r.returncode == 0 else None
        except Exception as e:
            print(f"  ⚠️ antiword falló para {fp.name}: {e} (instala antiword o usa KG_SOURCE='hybrid')")
            return None

    files = sorted(list(Path(raw_dir).glob("*.docx")) + list(Path(raw_dir).glob("*.doc")))
    if not files:
        raise FileNotFoundError(f"Sin .doc/.docx en {raw_dir}")

    texts, sources = [], []
    for fp in files:
        raw = _extract_docx(fp) if fp.suffix.lower() == ".docx" else _extract_doc(fp)
        if raw:
            texts.append(raw)
            sources.append(fp.stem)
    return texts, sources, f"originales .doc/.docx desde {raw_dir}"


_PREPS = {"chunks": _prepare_from_chunks, "hybrid": _prepare_hybrid, "original": _prepare_original}


def _family_of(source_id: str) -> str:
    """Familia = primer segmento numérico del doc_id (ej: '244170', '254275')."""
    head = source_id.split("__")[0]            # quita sufijos de chunk_id tipo __c0001
    return head.split("-")[0] if "-" in head else head


def _balanced_take(texts, sources, k):
    """Devuelve k elementos repartidos entre familias (round-robin)."""
    from collections import defaultdict
    by_family = defaultdict(list)
    for t, s in zip(texts, sources):
        by_family[_family_of(s)].append((t, s))
    families = sorted(by_family.keys())
    out_t, out_s = [], []
    idx = 0
    while len(out_t) < k:
        any_taken = False
        for fam in families:
            if idx < len(by_family[fam]):
                t, s = by_family[fam][idx]
                out_t.append(t); out_s.append(s)
                any_taken = True
                if len(out_t) >= k:
                    break
        idx += 1
        if not any_taken:
            break
    return out_t, out_s, families


def prepare_inputs(src: str, subset_size=SUBSET_SIZE):
    """Devuelve (textos, fuentes, descripcion) según el modo."""
    if src not in _PREPS:
        raise ValueError(f"KG_SOURCE inválido: {src}. Usa: {list(_PREPS)}")
    texts, sources, desc = _PREPS[src]()
    if subset_size is not None and subset_size < len(texts):
        if BALANCED_SAMPLING:
            texts, sources, fams = _balanced_take(texts, sources, subset_size)
            desc += f"  [balanceado entre familias: {', '.join(fams)}]"
        else:
            texts = texts[:subset_size]
            sources = sources[:subset_size]
    return texts, sources, desc


for _src in KG_SOURCES:
    try:
        _t, _s, _d = prepare_inputs(_src)
        print(f"[{_src:>8}] {len(_t)} elementos | {sum(len(t) for t in _t):,} chars | {_d}")
    except Exception as e:
        print(f"[{_src:>8}] ⚠️  No disponible: {e}")


[  chunks] 10 elementos | 4,516 chars | chunks de Chroma (tal cual)  [balanceado entre familias: 244170, 252562, 254275, Nota_Técnica_Ejecucion_Radon_SIKA]
[  hybrid] 10 elementos | 78,681 chars | actas reconstruidas desde Chroma (chunks agrupados por doc_id)  [balanceado entre familias: 244170, 252562, 254275, Nota_Técnica_Ejecucion_Radon_SIKA]
[original] 10 elementos | 127,593 chars | originales .doc/.docx desde /content/drive/MyDrive/RAG_UPC_Final_project  [balanceado entre familias: 244170, 252562, 254275, Nota_Técnica_Ejecucion_Radon_SIKA]


In [5]:
from kg_gen import KGGen

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY no encontrada — revisa tu .env")

kg = KGGen(model=KG_MODEL, temperature=0.0, api_key=api_key)
print(f"✅ KGGen inicializado con {KG_MODEL}")


✅ KGGen inicializado con openai/gpt-4o-mini


In [6]:
_subset_tag = f"sub{SUBSET_SIZE}" if SUBSET_SIZE else "full"


def graph_json_path(src: str) -> Path:
    do_cluster = KG_CLUSTER_BY_SOURCE.get(src, KG_CLUSTER)
    bal = "bal" if BALANCED_SAMPLING else "seq"
    clus = "cl" if do_cluster else "raw"
    return Path(GRAPH_DIR) / f"graph_actas_e5_{src}_{_subset_tag}_{bal}_{clus}.json"


def graph_to_dict(graph, src: str, texts, sources):
    return {
        "entities": sorted(list(graph.entities)),
        "edges": sorted(list(graph.edges)),
        "relations": [list(r) for r in graph.relations],
        "entity_clusters": {k: sorted(list(v)) for k, v in (graph.entity_clusters or {}).items()},
        "edge_clusters": {k: sorted(list(v)) for k, v in (graph.edge_clusters or {}).items()},
        "meta": {
            "model": KG_MODEL,
            "kg_source": src,
            "subset_size": SUBSET_SIZE,
            "balanced_sampling": BALANCED_SAMPLING,
            "n_inputs": len(texts),
            "sources": sources,
            "context": KG_CONTEXT,
            "cluster": KG_CLUSTER_BY_SOURCE.get(src, KG_CLUSTER),
        },
    }


def build_and_persist(src: str):
    """Genera el grafo elemento a elemento (con progress) y los agrega al final.

    En vez de pasar todo el corpus a kg.generate (que tarda 10+ min sin output),
    procesamos cada elemento por separado e imprimimos progreso. Luego kg.aggregate
    combina los grafos y kg.cluster hace entity resolution global UNA sola vez.
    """
    out = graph_json_path(src)
    if out.exists() and src not in FORCE_REGEN:
        print(f"[{src:>8}] 📁 ya existe en Drive → {out.name} (no se regenera)")
        with open(out, "r", encoding="utf-8") as f:
            return json.load(f)
    if src in FORCE_REGEN and out.exists():
        print(f"[{src:>8}] 🔁 FORCE_REGEN: se regenera aunque exista {out.name}")
    texts, sources, desc = prepare_inputs(src)
    if not texts:
        print(f"[{src:>8}] ⚠️  Sin textos, salto.")
        return None
    total_chars = sum(len(t) for t in texts)
    print(f"[{src:>8}] {len(texts)} elementos ({total_chars:,} chars) — generando elemento a elemento...")
    t0 = time.perf_counter()
    partial_graphs = []
    for i, txt in enumerate(texts, 1):
        ti = time.perf_counter()
        # cluster=False aquí: clusterizamos UNA sola vez al final, más barato y mejor
        g_i = kg.generate(input_data=txt, context=KG_CONTEXT, chunk_size=KG_CHUNK_SIZE, cluster=False)
        partial_graphs.append(g_i)
        elapsed = time.perf_counter() - t0
        eta = elapsed / i * (len(texts) - i)
        print(
            f"[{src:>8}] {i:>3}/{len(texts)}  {sources[i-1][:40]:<40}  "
            f"+{len(g_i.relations):>4} triples  ({time.perf_counter()-ti:.1f}s)  "
            f"ETA {eta/60:.1f} min"
        )
    print(f"[{src:>8}] Agregando {len(partial_graphs)} grafos parciales...")
    graph = kg.aggregate(partial_graphs)
    do_cluster = KG_CLUSTER_BY_SOURCE.get(src, KG_CLUSTER)
    if do_cluster and graph.relations:
        print(f"[{src:>8}] Clustering global (entity resolution)...")
        graph = kg.cluster(graph, context=KG_CONTEXT)
    else:
        print(f"[{src:>8}] (clustering saltado para esta fuente — triples crudos)")
    elapsed = time.perf_counter() - t0
    print(f"[{src:>8}] ✅ {len(graph.relations)} triples totales en {elapsed/60:.1f} min")
    gd = graph_to_dict(graph, src, texts, sources)
    with open(out, "w", encoding="utf-8") as f:
        json.dump(gd, f, ensure_ascii=False, indent=2)
    print(f"[{src:>8}] 💾 {out}")
    return gd


graphs_by_source = {}
for _src in KG_SOURCES:
    try:
        gd = build_and_persist(_src)
        if gd is not None:
            graphs_by_source[_src] = gd
    except Exception as e:
        print(f"[{_src:>8}] ❌ Error: {e}")

print(f"\nGrafos disponibles: {list(graphs_by_source)}")

# Carga el grafo primario en variables globales para las secciones interactivas 3-7
if KG_SOURCE_PRIMARY in graphs_by_source:
    graph_dict = graphs_by_source[KG_SOURCE_PRIMARY]
    print(f"Grafo primario para secciones 3-7: {KG_SOURCE_PRIMARY} ({len(graph_dict['relations'])} triples)")
elif graphs_by_source:
    graph_dict = next(iter(graphs_by_source.values()))
    print(f"⚠️  {KG_SOURCE_PRIMARY} no disponible, usando: {graph_dict['meta']['kg_source']}")
else:
    raise RuntimeError("No se pudo generar ningún grafo.")


[  chunks] 📁 ya existe en Drive → graph_actas_e5_chunks_sub10_bal_cl.json (no se regenera)
[  hybrid] 📁 ya existe en Drive → graph_actas_e5_hybrid_sub10_bal_cl.json (no se regenera)
[original] 📁 ya existe en Drive → graph_actas_e5_original_sub10_bal_raw.json (no se regenera)

Grafos disponibles: ['chunks', 'hybrid', 'original']
Grafo primario para secciones 3-7: hybrid (312 triples)


### 2.1 (Opcional) Subir grafo a GCS

Mismo bucket que el Chroma (`GCP_BUCKET_NAME`), prefijo `graph/`. Salta esta celda si no estás desplegando.


In [ ]:
def upload_graph_to_gcs(local_path: Path, bucket_name: str, prefix: str = "graph"):
    if not bucket_name:
        print("⚠️  GCS_BUCKET vacío — exporta GCP_BUCKET_NAME en .env para subir")
        return
    from google.cloud import storage
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob_name = f"{prefix}/{local_path.name}"
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(str(local_path))
    print(f"☁️  Subido: gs://{bucket_name}/{blob_name}")


# Descomenta para subir:
# upload_graph_to_gcs(GRAPH_JSON_PATH, GCS_BUCKET, GCS_GRAPH_PREFIX)


### 2.2 Visualización del grafo (nodos y relaciones)

Los grafos se guardan como **JSON** en:

| Entorno | Ruta |
|---------|------|
| **Colab / Drive** | `My Drive/RAG_UPC_Final_project/graph/graph_actas_e5_{fuente}_sub10_bal_{cl\|raw}.json` |
| **Local** | `data/graph/` (misma convención de nombre) |

Cada archivo contiene:
- **`entities`** — nodos (entidades extraídas por kg-gen)
- **`relations`** — aristas `[sujeto, relación, objeto]` (triples)
- **`meta`** — fuente (`chunks` / `hybrid` / `original`), subset, etc.

Ajusta `VIZ_SOURCE`, `VIZ_SEARCH` y `VIZ_MAX_EDGES` en la celda siguiente.

In [ ]:
%pip install -q pyvis

import pandas as pd
from collections import Counter
from IPython.display import HTML, display

# ─── Parámetros de visualización ─────────────────────────────────────
VIZ_SOURCE = "original"      # "chunks" | "hybrid" | "original"
VIZ_MAX_EDGES = 120          # límite de triples (el grafo completo puede ser ilegible)
VIZ_SEARCH = None            # ej. "talud", "megafonía", "UTE" — filtra por texto en triple
VIZ_HEIGHT = "650px"

print(f"📂 GRAPH_DIR: {GRAPH_DIR}\n")
for _src in KG_SOURCES:
    _p = graph_json_path(_src)
    print(f"  [{'✓' if _p.exists() else '✗'}] {_p.name}")


def load_graph_viz(src: str) -> dict:
    """Carga grafo desde memoria (sec 2) o desde JSON en Drive/local."""
    if "graphs_by_source" in dir() and src in graphs_by_source:
        print(f"📦 Usando grafo en memoria: {src}")
        return graphs_by_source[src]
    path = graph_json_path(src)
    if not path.exists():
        raise FileNotFoundError(f"No existe {path}. Ejecuta la sec 2 primero.")
    print(f"📂 Cargando: {path}")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def print_graph_stats(gd: dict):
    rels = [tuple(r) for r in gd["relations"]]
    nodes = set()
    for s, r, o in rels:
        nodes.add(s)
        nodes.add(o)
    rel_types = Counter(r for _, r, _ in rels)
    print(f"\n📊 Estadísticas ({gd.get('meta', {}).get('kg_source', VIZ_SOURCE)}):")
    print(f"   Nodos (entidades):     {len(nodes)}")
    print(f"   Aristas (triples):     {len(rels)}")
    print(f"   Tipos de relación:     {len(rel_types)}")
    print(f"   Top relaciones:        {rel_types.most_common(5)}")
    meta = gd.get("meta", {})
    if meta:
        print(f"   Subset: {meta.get('subset_size')} | cluster: {meta.get('cluster')}")


def visualize_kg(gd: dict, max_edges=VIZ_MAX_EDGES, search=VIZ_SEARCH, height=VIZ_HEIGHT):
    from pyvis.network import Network

    rels = [tuple(r) for r in gd["relations"]]
    if search:
        st = search.lower()
        rels = [t for t in rels if st in t[0].lower() or st in t[2].lower() or st in t[1].lower()]
        print(f"\n🔍 Filtro '{search}': {len(rels)} triples")
    total = len(rels)
    if len(rels) > max_edges:
        rels = rels[:max_edges]
        print(f"⚠️  Mostrando {max_edges}/{total} triples. Sube VIZ_MAX_EDGES o usa VIZ_SEARCH.")

    net = Network(
        height=height, width="100%", bgcolor="#fafafa", font_color="#222222",
        directed=True, notebook=True,
    )
    net.barnes_hut(gravity=-12000, central_gravity=0.35, spring_length=150, spring_strength=0.04)

    node_deg = Counter()
    for s, r, o in rels:
        node_deg[s] += 1
        node_deg[o] += 1

    added = set()
    for s, r, o in rels:
        for n in (s, o):
            if n not in added:
                lbl = n if len(n) <= 40 else n[:37] + "…"
                net.add_node(n, label=lbl, title=n, value=max(node_deg[n], 1))
                added.add(n)
        elbl = r if len(r) <= 22 else r[:19] + "…"
        net.add_edge(s, o, title=r, label=elbl)

    out_html = Path("graph_viz_cosora.html")
    net.save_graph(str(out_html))
    print(f"\n💾 HTML interactivo: {out_html.resolve()}")
    if IN_COLAB:
        print("   Descargar en Colab: from google.colab import files; files.download('graph_viz_cosora.html')")
    display(HTML(out_html.read_text(encoding="utf-8")))


# ─── Ejecutar ────────────────────────────────────────────────────────
_gd_viz = load_graph_viz(VIZ_SOURCE)
print_graph_stats(_gd_viz)

print("\n📋 Muestra de triples (tabla):")
display(
    pd.DataFrame(_gd_viz["relations"][:15], columns=["sujeto", "relación", "objeto"])
)

print("\n🕸️  Grafo interactivo (arrastra nodos, zoom con rueda):")
visualize_kg(_gd_viz)

## 3. Retrieval del grafo

Embebemos cada triple (`"sujeto relación objeto"`) con **E5** una sola vez y buscamos por similitud coseno contra la query. Misma representación semántica que usa el dense retriever de chunks.


In [8]:
def triple_to_text(rel):
    s, r, o = rel
    return f"{s} {r} {o}"


relations = [tuple(r) for r in graph_dict["relations"]]
triple_texts = [triple_to_text(r) for r in relations]

print(f"Embebiendo {len(triple_texts)} triples con E5...")
t0 = time.perf_counter()
triple_vecs = embedder.embed_batch(triple_texts, is_query=False, batch_size=32)
triple_mat = np.array(triple_vecs, dtype=np.float32)
# Normalizar para coseno
triple_norm = triple_mat / (np.linalg.norm(triple_mat, axis=1, keepdims=True) + 1e-12)
print(f"✅ Triples embebidos en {time.perf_counter() - t0:.1f}s — shape={triple_mat.shape}")


Embebiendo 312 triples con E5...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Triples embebidos en 0.7s — shape=(312, 768)


In [9]:
def retrieve_graph(query, k=GRAPH_TOP_K):
    """Devuelve los k triples más relevantes para la query (cosine con E5)."""
    q_vec = np.array(embedder.embed_one(query, is_query=True), dtype=np.float32)
    q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-12)
    sims = triple_norm @ q_vec
    top_idx = np.argsort(-sims)[:k]
    return [
        {
            "triple": relations[i],
            "text": triple_texts[i],
            "score": float(sims[i]),
            "rank_graph": rank,
        }
        for rank, i in enumerate(top_idx)
    ]


# Sanity check
sample = retrieve_graph("incidencias en el talud", k=5)
for t in sample:
    print(f"  cos={t['score']:.3f} | {t['text']}")


  cos=0.849 | DF envía en resultado de la verificación de documentación aportada por la UTE sobre la estabilidad del talud
  cos=0.827 | tapiado afecta a huecos de fachada
  cos=0.820 | DO ejecuta Tabiques de pladur
  cos=0.816 | UTE informa gestión de solicitud
  cos=0.815 | UTE atenderá correcciones indicadas en los informes de revisión


## 4. Fusión RRF triple: dense + BM25 + graph

El grafo no recupera *chunks*, recupera *hechos*. Lo combinamos de dos formas:

- **Boost de chunks**: para cada triple top-K, localizamos los chunks cuyo texto contiene al sujeto u objeto del triple y aportamos un rank extra a la RRF.
- **Contexto adicional en el prompt**: los triples top-K se pasan al LLM como sección "Hechos relacionados".


In [10]:
_NORMALIZE_RE = re.compile(r"[^\wáéíóúñü]+", re.IGNORECASE)


def _tokenize_match(s: str) -> set[str]:
    s = s.lower()
    return set(t for t in _NORMALIZE_RE.split(s) if t and len(t) > 2 and t not in STOPWORDS_ES)


def _match_substring(triple, doc):
    s, _, o = triple
    d_low = doc.lower()
    return s.lower() in d_low or o.lower() in d_low


def _match_wordset(triple, doc, min_overlap=MATCH_MIN_OVERLAP):
    """Exige solape de palabras (no substring): el sujeto Y el objeto
    deben tener >=min_overlap de sus palabras significativas en el chunk."""
    s, _, o = triple
    doc_words = _tokenize_match(doc)
    if not doc_words:
        return False
    s_words = _tokenize_match(s)
    o_words = _tokenize_match(o)

    def overlap_ok(words):
        if not words:
            return False
        common = words & doc_words
        return len(common) / len(words) >= min_overlap

    # AND: ambos lados del triple deben matchear (no solo uno)
    return overlap_ok(s_words) and overlap_ok(o_words)


def chunks_matching_triple(triple, docs, metas, strategy=None):
    """Devuelve índices de chunks que matchean el triple según `strategy`.

    - "substring": el sujeto O el objeto del triple aparece como substring (legacy, laxo).
    - "wordset":   AMBOS lados deben tener >=MATCH_MIN_OVERLAP de sus palabras significativas
                   en el chunk. Más estricto, evita boosts espurios (D1).
    """
    strategy = strategy or MATCH_STRATEGY
    matcher = _match_substring if strategy == "substring" else _match_wordset
    return [i for i, d in enumerate(docs) if matcher(triple, d)]


def rrf_fusion_graph(dense, bm25, graph_triples, docs, metas, rrf_k=60, top_n=10, min_cos=GRAPH_MIN_COS):
    """Fusión RRF dense + BM25 + grafo.

    El grafo SOLO boostea chunks si el triple supera `min_cos`. Sin umbral, queries
    cuyo tema no está en el grafo recibirían boost de triples mediocres y degradarían
    los resultados (verificado en sub2: hybrid empuja chunks de cubiertas para queries
    sobre talud).
    """
    scores = {}
    for item in dense:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0, "graph_boost": 0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_dense"])
    for item in bm25:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0, "graph_boost": 0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_bm25"])
    # Graph boost con umbral
    used = 0
    for t in graph_triples:
        if t["score"] < min_cos:
            continue
        used += 1
        idxs = chunks_matching_triple(t["triple"], docs, metas)
        for i in idxs:
            cid = metas[i]["chunk_id"]
            scores.setdefault(cid, {"text": docs[i], "meta": metas[i], "score": 0.0, "graph_boost": 0})
            scores[cid]["score"] += 1.0 / (rrf_k + t["rank_graph"])
            scores[cid]["graph_boost"] += 1
    # Devuelve también cuántos triples superaron el umbral (para diagnóstico)
    ranked = sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_n]
    if ranked and used == 0:
        ranked[0]["_no_graph_boost"] = True
    return ranked


def retrieve_graph_rag(query, top_n=TOP_N):
    """Retrieval con BOOST de grafo: dense + BM25 + grafo en RRF."""
    d = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    b = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    g = retrieve_graph(query, k=GRAPH_TOP_K)
    return rrf_fusion_graph(d, b, g, all_docs, all_metas, rrf_k=RRF_K, top_n=top_n), g


def retrieve_graph_context_only(query, top_n=TOP_N):
    """Retrieval sin boost (igual al baseline) pero devuelve también los triples top-K
    para usarlos como contexto adicional en el prompt. Aísla el efecto del grafo en
    el prompt vs el efecto del boost en el ranking."""
    chunks = retrieve_baseline(query, top_n=top_n)
    triples = retrieve_graph(query, k=GRAPH_TOP_K)
    return chunks, triples


## 5. Prompt builder enriquecido y `ask_cosora_graph`


In [11]:
def build_prompt_graph(query, chunks, graph_triples):
    """Prompt enriquecido con triples del grafo.

    Nota: probamos una versión 'strict_triples' con instrucciones negativas
    ("PROHIBIDO citar triples...") y EMPEORÓ la métrica `citation` con
    gpt-4o-mini (1.10 → 0.90 global). Revertido al prompt original suave.
    """
    context_blocks = []
    for i, ch in enumerate(chunks, 1):
        doc_id = ch["meta"]["doc_id"]
        text = strip_doc_prefix(ch["text"], EMBED_BACKEND)
        boost = ch.get("graph_boost", 0)
        tag = f" [grafo×{boost}]" if boost else ""
        context_blocks.append(f"[Fragmento {i} - Fuente: {doc_id}{tag}]\n{text}")
    context = "\n\n".join(context_blocks)

    facts = "\n".join(f"- {t['text']}" for t in graph_triples) if graph_triples else "(ninguno)"

    return f"""Eres COSORA, un asistente técnico especializado en análisis de actas de obra ferroviaria en España.

REGLAS ESTRICTAS:
1. Responde ÚNICAMENTE con información del CONTEXTO o de los HECHOS RELACIONADOS.
2. Si la información no está, dilo explícitamente.
3. Cita siempre la fuente entre paréntesis: (Fuente: nombre_del_acta).
4. Los HECHOS RELACIONADOS son triples extraídos automáticamente; úsalos como pistas, no como fuente primaria.

VOCABULARIO: DF=Dirección Facultativa, UTE=Unión Temporal de Empresas, DEO=Director de Ejecución de Obra, DO=Director de Obra.

=== HECHOS RELACIONADOS (grafo) ===
{facts}

=== CONTEXTO (fragmentos) ===
{context}

=== PREGUNTA ===
{query}

=== RESPUESTA ==="""


def generate_answer(prompt, model="gpt-4o-mini"):
    from openai import OpenAI
    api_key = os.getenv("OPENAI_API_KEY")
    client = OpenAI(api_key=api_key)
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return resp.choices[0].message.content


def filter_chunks(chunks):
    if not chunks or chunks[0]["score"] < RRF_MIN_SCORE:
        return None
    max_score = chunks[0]["score"]
    return [c for c in chunks if c["score"] >= max_score * SCORE_RATIO]


def ask_cosora_graph(query, verbose=True):
    chunks, triples = retrieve_graph_rag(query, top_n=TOP_N)
    filtered = filter_chunks(chunks)
    if filtered is None:
        return "No se encontró información suficientemente relevante en las actas."
    prompt = build_prompt_graph(query, filtered, triples)
    answer = generate_answer(prompt)
    if verbose:
        print(f"Query: {query}")
        print(f"\nTriples top-{len(triples)}:")
        for t in triples:
            print(f"  cos={t['score']:.3f} | {t['text']}")
        print(f"\nChunks ({len(filtered)}):")
        for c in filtered:
            boost = f" g×{c.get('graph_boost', 0)}" if c.get('graph_boost') else ""
            print(f"  - [{c['meta']['doc_id']}] score={c['score']:.4f}{boost}")
        print(f"\nRespuesta:\n{answer}\n" + "-"*70)
    return answer


# Test rápido
_ = ask_cosora_graph("¿Qué se decidió sobre el talud?", verbose=True)


Query: ¿Qué se decidió sobre el talud?

Triples top-5:
  cos=0.831 | DF envía en resultado de la verificación de documentación aportada por la UTE sobre la estabilidad del talud
  cos=0.781 | UTE informa que ha realizado la gestión de solicitud de Certificado de Aceptación de residuos
  cos=0.779 | DF solicita a la UTE el estado de la gestión
  cos=0.774 | DEO informa canalización de la red de baldeo
  cos=0.774 | DF hace elección de muestras de deployé

Chunks (7):
  - [254275-DO-AVO-15-V07] score=0.0333
  - [254275-DO-AVO-14-V07] score=0.0325
  - [254275-DO-AVO-16-V07] score=0.0323
  - [254275-DO-AVO-16-V07] score=0.0320
  - [254275-DO-AVO-14-V07] score=0.0305
  - [254275-DO-AVO-15-V07] score=0.0295
  - [254275-DO-AVO-22-V01] score=0.0291

Respuesta:
Se decidió que, para cerrar la verificación de la ejecución de hinca de perfiles en el talud, se debe confirmar las profundidades reales de hinca (empotramiento en estratos competentes), separaciones y orientación respecto al talud calcu

## 6. Evaluación LLM-as-Judge — baseline vs Graph RAG

Mismo juez (`gpt-4o-mini`) y mismas queries que en `query_pipeline.ipynb`. Comparamos:

- **baseline**: dense (E5) + BM25 + RRF
- **graph**: dense + BM25 + grafo + RRF triple

Métricas: Precision@K, MRR, NDCG@K, MAP.

> Coste: ~0,02 USD para 8 queries × 2 variantes × 5 chunks.


In [12]:
import pandas as pd
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)
LLM_JUDGE_MODEL = "gpt-4o-mini"

BENCHMARK_QUERIES_FACTUAL = [
    "¿Qué se decidió sobre el talud?",
    "¿Cuál es el estado del camino provisional?",
    "¿Qué incidencias AR-29 aparecen?",
    "¿Qué responsable está asignado a las acciones sobre el talud?",
    "¿Qué se acordó sobre hormigonado de zapatas?",
    "Estado de las instalaciones de megafonía",
    "¿Qué documentación debe aportar la UTE sobre estabilidad?",
    # Añadidas de docs/queries.md (Factual directa)
    "¿Cuál es la función de la sigla DEO en un acta de visita de obra?",
    "¿Cuántos días hay de plazo para devolver el acta firmada?",
    "¿Qué modelo de luminaria debe instalarse en los báculos del andén?",
    "¿Qué certificados de material debe aportar la UTE sobre las hincas de perfiles?",
    "¿A qué altura respecto al suelo se van a instalar las unidades exteriores de climatización?",
]

# Queries transversales/analíticas — extraídas de docs/queries.md (BLOQUES 1, 2, 4, 5, 7, 11).
# Es donde se espera que el grafo aporte valor frente al baseline.
BENCHMARK_QUERIES_TRANSVERSAL = [
    "¿Cuáles son las incidencias más frecuentes en todas las actas?",
    "¿Qué elementos constructivos presentan más incidencias?",
    "¿Qué problemas pueden afectar a la seguridad de la explotación ferroviaria?",
    "Agrupa todas las incidencias detectadas en las actas y clasifícalas por tipo",
    "¿Cuáles son las acciones pendientes más frecuentes en las actas?",
    # Añadidas
    "¿Cuáles son los problemas más frecuentes en instalaciones eléctricas y luminarias?",
    "Lista todas las solicitudes realizadas a la constructora",
    "¿Qué incumplimientos normativos se detectan en el conjunto de actas?",
    "¿Qué tipo de tareas suelen quedar sin resolver?",
]
# Excluida deliberadamente: "Compara las diferentes obras en función del número de incidencias"
# → ningún sistema la puede contestar (las actas no comparan obras explícitamente) y daba
#   NDCG=0 a todas las variantes, distorsionando la media.

BENCHMARK_QUERIES = BENCHMARK_QUERIES_FACTUAL + BENCHMARK_QUERIES_TRANSVERSAL
print(f"Queries totales: {len(BENCHMARK_QUERIES)} ({len(BENCHMARK_QUERIES_FACTUAL)} factual + {len(BENCHMARK_QUERIES_TRANSVERSAL)} transversal)")
EVAL_K = max(EVAL_K_LIST)   # juzgamos los top-max(K) y derivamos métricas por cada K


JUDGE_DEBUG = False  # Pon True para imprimir respuestas crudas de las primeras 3 llamadas
_judge_debug_remaining = 3


def judge_relevance(query: str, chunk_text: str) -> int:
    global _judge_debug_remaining
    system = (
        "Eres un evaluador de relevancia para un sistema de búsqueda sobre actas de obra "
        "ferroviaria. Tu única tarea: emitir un dígito (0, 1 o 2) que valore si el FRAGMENTO "
        "es relevante para la PREGUNTA. Sin texto adicional."
    )
    user = (
        f"PREGUNTA: {query}\n\n"
        f"FRAGMENTO:\n{chunk_text[:600]}\n\n"
        "Criterios:\n"
        "0 = el fragmento no aporta nada a la pregunta\n"
        "1 = menciona el tema pero no contesta\n"
        "2 = contiene información que responde directa o sustancialmente a la pregunta\n\n"
        "Responde SOLO con el dígito 0, 1 o 2."
    )
    try:
        resp = client.chat.completions.create(
            model=LLM_JUDGE_MODEL,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            temperature=0,
            max_tokens=2,
        )
        raw = resp.choices[0].message.content or ""
    except Exception as e:
        if JUDGE_DEBUG:
            print(f"  judge_relevance EXC: {type(e).__name__}: {e}")
        return -1  # marcador de error, no se cuenta como irrelevante
    ans = raw.strip()
    if JUDGE_DEBUG and _judge_debug_remaining > 0:
        print(f"  judge raw={raw!r}  → ans={ans!r}")
        _judge_debug_remaining -= 1
    # Toma el PRIMER dígito 0/1/2 del string (la respuesta debería empezar por él)
    for ch in ans:
        if ch in "012":
            return int(ch)
    return -1  # respuesta no parseable, no se cuenta


def dcg_at_k(rels, k):
    return sum(r / np.log2(i + 2) for i, r in enumerate(rels[:k]))

def ndcg_at_k(rels, k):
    dcg = dcg_at_k(rels, k)
    ideal = dcg_at_k(sorted(rels, reverse=True), k)
    return dcg / ideal if ideal > 0 else 0.0

def average_precision(rels):
    rel_count, psum = 0, 0.0
    for i, r in enumerate(rels):
        if r >= 1:
            rel_count += 1
            psum += rel_count / (i + 1)
    return psum / rel_count if rel_count else 0.0

def mrr(rels):
    for i, r in enumerate(rels):
        if r >= 1:
            return 1.0 / (i + 1)
    return 0.0


def _clean_rels(rels):
    """Sustituye -1 (error de juez) por 0 para que las métricas no se rompan."""
    return [max(r, 0) for r in rels]


def eval_variant(name, retrieve_fn, queries, k_list=EVAL_K_LIST, judge_fn=None):
    """Evalúa una variante recuperando max(k_list) chunks y calculando métricas para cada K.

    judge_fn: callable(query, chunk_text) -> int 0/1/2/-1. Default = judge_relevance (gpt-4o-mini).
    """
    judge_fn = judge_fn or judge_relevance
    k_max = max(k_list)
    judgments = []
    n_errors = 0
    for q in queries:
        hits = retrieve_fn(q, k_max)
        rels = []
        for h in hits:
            txt = strip_doc_prefix(h["text"], EMBED_BACKEND)
            r = judge_fn(q, txt)
            if r == -1:
                n_errors += 1
            rels.append(r)
        clean = _clean_rels(rels)
        entry = {"query": q, "relevances": rels, "clean_relevances": clean}
        for k in k_list:
            r_k = clean[:k]
            entry[f"precision@{k}"] = sum(1 for r in r_k if r >= 1) / k
            entry[f"ndcg@{k}"] = ndcg_at_k(r_k, k)
        entry["mrr"] = mrr(clean)
        entry["ap"] = average_precision(clean)
        judgments.append(entry)
    if n_errors:
        print(f"  ⚠️  {name}: {n_errors} juicios no parseables (ver JUDGE_DEBUG=True)")
    summary = {"variant": name}
    for k in k_list:
        summary[f"Precision@{k}"] = np.mean([j[f"precision@{k}"] for j in judgments])
        summary[f"NDCG@{k}"] = np.mean([j[f"ndcg@{k}"] for j in judgments])
    summary["MRR"] = np.mean([j["mrr"] for j in judgments])
    summary["MAP"] = np.mean([j["ap"] for j in judgments])
    return summary, judgments


# Cross-judge opcional (otro modelo como juez para validar)
def _make_judge(model_name: str):
    def _judge(query: str, chunk_text: str) -> int:
        system = (
            "Eres un evaluador de relevancia para un sistema de búsqueda sobre actas de obra "
            "ferroviaria. Tu única tarea: emitir un dígito (0, 1 o 2)."
        )
        user = (
            f"PREGUNTA: {query}\n\n"
            f"FRAGMENTO:\n{chunk_text[:600]}\n\n"
            "0 = irrelevante, 1 = parcial, 2 = muy relevante.\n"
            "Responde SOLO con el dígito."
        )
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
                temperature=0, max_tokens=2,
            )
            raw = resp.choices[0].message.content or ""
        except Exception:
            return -1
        for ch in raw.strip():
            if ch in "012":
                return int(ch)
        return -1
    return _judge


def retrieve_baseline_k(q, k):
    return retrieve_baseline(q, top_n=k)

def retrieve_graph_k(q, k):
    chunks, _ = retrieve_graph_rag(q, top_n=k)
    return chunks


print("Evaluando baseline...")
base_summary, base_detail = eval_variant("baseline", retrieve_baseline_k, BENCHMARK_QUERIES)
print("Evaluando graph (con boost)...")
graph_summary, graph_detail = eval_variant("graph", retrieve_graph_k, BENCHMARK_QUERIES)

df = pd.DataFrame([base_summary, graph_summary]).set_index("variant")
print("\n🏆 Resultados (multi-K):")
display(df)

# Delta por query — usamos K=5 como referencia para el ordering
K_REF = EVAL_K_LIST[0]
delta_rows = []
for i, q in enumerate(BENCHMARK_QUERIES):
    delta_rows.append({
        "query": q[:50],
        f"base_NDCG@{K_REF}": base_detail[i][f"ndcg@{K_REF}"],
        f"graph_NDCG@{K_REF}": graph_detail[i][f"ndcg@{K_REF}"],
    })
df_delta = pd.DataFrame(delta_rows)
df_delta["Δ NDCG"] = df_delta[f"graph_NDCG@{K_REF}"] - df_delta[f"base_NDCG@{K_REF}"]
display(df_delta.sort_values("Δ NDCG", ascending=False))


Queries totales: 21 (12 factual + 9 transversal)
Evaluando baseline...
Evaluando graph (con boost)...

🏆 Resultados (multi-K):


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
baseline,0.771429,0.901833,0.633333,0.893126,0.900794,0.852504
graph,0.695238,0.781099,0.623810,0.805745,0.762698,0.752360


,query,base_NDCG@5,graph_NDCG@5,Δ NDCG
20,¿Qué tipo de tareas suelen quedar sin resolver?,0.501266,0.904717,0.403451
16,¿Cuáles son las acciones pendientes más frecue...,0.852928,1.000000,0.147072
2,¿Qué incidencias AR-29 aparecen?,0.867087,1.000000,0.132913
19,¿Qué incumplimientos normativos se detectan en...,0.500000,0.543771,0.043771
0,¿Qué se decidió sobre el talud?,1.000000,1.000000,0.000000
14,¿Qué problemas pueden afectar a la seguridad d...,0.982892,0.982892,0.000000
4,¿Qué se acordó sobre hormigonado de zapatas?,1.000000,1.000000,0.000000
1,¿Cuál es el estado del camino provisional?,0.906528,0.906528,0.000000
8,¿Cuántos días hay de plazo para devolver el ac...,1.000000,1.000000,0.000000
13,¿Qué elementos constructivos presentan más inc...,1.000000,1.000000,0.000000


## 7. Pruebas interactivas


In [13]:
MIS_PREGUNTAS = [
    "¿Qué se decidió sobre el talud?",
    "¿Cuál es el estado del camino provisional?",
    "¿Qué incidencias aparecen relacionadas con instalaciones eléctricas?",
]

for q in MIS_PREGUNTAS:
    print("="*80)
    print(f"👤 {q}")
    print("-"*80)
    resp = ask_cosora_graph(q, verbose=False)
    print(f"🤖 {resp}")


👤 ¿Qué se decidió sobre el talud?
--------------------------------------------------------------------------------
🤖 Se decidió que, para cerrar la verificación de la ejecución de hinca de perfiles en el talud, faltaba confirmar las profundidades reales de hinca, las separaciones y la orientación respecto al talud calculado. Además, se solicitó la aportación de los Certificados de material de hinca (tipo de carril, acero, características) (Fuente: 254275-DO-AVO-15-V07).
👤 ¿Cuál es el estado del camino provisional?
--------------------------------------------------------------------------------
🤖 El estado del camino provisional indica que se está continuando con el rebaje del nivel de tierras, lo que mejora las condiciones de seguridad y estabilidad del entorno, ya que disminuye el empuje de tierras hacia las parcelas. Sin embargo, también se ha constatado la presencia de tierras desprendidas y acumuladas en contacto directo con los muros de los patios colindantes de las viviendas (Fue

## 8. Comparativa global — baseline vs grafo (chunks / hybrid / original)

Evalúa **baseline + los 3 grafos que generó la sección 2** sobre las mismas queries de benchmark. Si alguno falló, se omite y se sigue.

> Coste: ~0,02 USD por cada variante de grafo presente (8 queries × 5 chunks × `gpt-4o-mini`).


In [14]:
def prepare_variant(gd):
    """Embebe los triples de un grafo (dict) para usarlos en retrieval."""
    rels = [tuple(r) for r in gd["relations"]]
    texts = [f"{s} {r} {o}" for s, r, o in rels]
    if not texts:
        return None
    vecs = embedder.embed_batch(texts, is_query=False, batch_size=32)
    mat = np.array(vecs, dtype=np.float32)
    norm = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)
    return {"meta": gd.get("meta", {}), "relations": rels, "texts": texts, "norm": norm}


def retrieve_graph_variant(query, variant, k=GRAPH_TOP_K):
    q = np.array(embedder.embed_one(query, is_query=True), dtype=np.float32)
    q = q / (np.linalg.norm(q) + 1e-12)
    sims = variant["norm"] @ q
    top = np.argsort(-sims)[:k]
    return [
        {"triple": variant["relations"][i], "text": variant["texts"][i], "score": float(sims[i]), "rank_graph": rk}
        for rk, i in enumerate(top)
    ]


def retrieve_graph_rag_with(query, variant, top_n=TOP_N):
    d = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    b = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    g = retrieve_graph_variant(query, variant, k=GRAPH_TOP_K)
    return rrf_fusion_graph(d, b, g, all_docs, all_metas, rrf_k=RRF_K, top_n=top_n)


loaded_variants = {}
for src, gd in graphs_by_source.items():
    v = prepare_variant(gd)
    if v is not None:
        loaded_variants[src] = v
        print(f"[{src:>8}] {len(v['relations'])} triples embebidos")

print(f"\nVariantes a comparar: {list(loaded_variants)}")


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

[  chunks] 47 triples embebidos


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

[  hybrid] 312 triples embebidos


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

[original] 426 triples embebidos

Variantes a comparar: ['chunks', 'hybrid', 'original']


In [15]:
all_summaries = []
all_details = {}

print("Evaluando baseline (dense + BM25)...")
base_s, base_d = eval_variant("baseline", retrieve_baseline_k, BENCHMARK_QUERIES)
all_summaries.append(base_s)
all_details["baseline"] = base_d

# Por cada grafo: dos variantes
#   - boost:     dense + BM25 + grafo en RRF (lo que veníamos haciendo)
#   - context:   dense + BM25 normal, triples sólo van al prompt (aísla el efecto del boost)
for src, variant in loaded_variants.items():
    # variante con BOOST
    name_boost = f"graph_{src}_boost"
    print(f"Evaluando {name_boost}...")
    fn_b = lambda q, k, _v=variant: retrieve_graph_rag_with(q, _v, top_n=k)
    s, d = eval_variant(name_boost, fn_b, BENCHMARK_QUERIES)
    all_summaries.append(s)
    all_details[name_boost] = d

    # variante sin BOOST (sólo contexto) — usa baseline para recuperar
    name_ctx = f"graph_{src}_ctxonly"
    print(f"Evaluando {name_ctx} (sin boost, triples sólo en prompt)...")
    fn_c = lambda q, k: retrieve_baseline_k(q, k)
    s, d = eval_variant(name_ctx, fn_c, BENCHMARK_QUERIES)
    all_summaries.append(s)
    all_details[name_ctx] = d

df_cmp = pd.DataFrame(all_summaries).set_index("variant")
print("\n🏆 Comparativa global (multi-K):")
display(df_cmp)

# Delta NDCG@K_REF por query — sólo para variantes con boost (las ctxonly son idénticas al baseline)
K_REF = EVAL_K_LIST[0]
delta_cols = {"query": [q[:50] for q in BENCHMARK_QUERIES]}
delta_cols[f"base_NDCG@{K_REF}"] = [d[f"ndcg@{K_REF}"] for d in all_details["baseline"]]
for src in loaded_variants:
    name = f"graph_{src}_boost"
    delta_cols[f"{name}_NDCG"] = [d[f"ndcg@{K_REF}"] for d in all_details[name]]
    delta_cols[f"Δ_{src}_boost"] = [
        all_details[name][i][f"ndcg@{K_REF}"] - all_details["baseline"][i][f"ndcg@{K_REF}"]
        for i in range(len(BENCHMARK_QUERIES))
    ]
df_delta = pd.DataFrame(delta_cols)
print(f"\nΔ NDCG@{K_REF} por query (boost - baseline):")
display(df_delta)

# Ganador por métrica
print("\n📊 Ganador por métrica (global):")
for col in [c for c in df_cmp.columns]:
    winner = df_cmp[col].idxmax()
    print(f"  {col:>15} → {winner} ({df_cmp.loc[winner, col]:.3f})")


def _summary_subset(name, details, queries_subset, k_list=EVAL_K_LIST):
    idxs = [i for i, q in enumerate(BENCHMARK_QUERIES) if q in queries_subset]
    if not idxs:
        return None
    row = {"variant": name}
    for k in k_list:
        row[f"Precision@{k}"] = np.mean([details[i][f"precision@{k}"] for i in idxs])
        row[f"NDCG@{k}"] = np.mean([details[i][f"ndcg@{k}"] for i in idxs])
    row["MRR"] = np.mean([details[i]["mrr"] for i in idxs])
    row["MAP"] = np.mean([details[i]["ap"] for i in idxs])
    return row


print("\n🎯 Subconjunto FACTUAL (lookup específico):")
rows = [_summary_subset(n, all_details[n], BENCHMARK_QUERIES_FACTUAL) for n in all_details]
display(pd.DataFrame([r for r in rows if r]).set_index("variant"))

print("\n🌐 Subconjunto TRANSVERSAL (agregadas/analíticas — donde el grafo debe ayudar):")
rows = [_summary_subset(n, all_details[n], BENCHMARK_QUERIES_TRANSVERSAL) for n in all_details]
display(pd.DataFrame([r for r in rows if r]).set_index("variant"))

# ─── Cross-judge opcional ──────────────────────────────────────────────────
if CROSS_JUDGE_MODEL:
    print(f"\n🧑‍⚖️ Cross-judge con {CROSS_JUDGE_MODEL} (validación independiente)...")
    cross_judge = _make_judge(CROSS_JUDGE_MODEL)
    cross_summaries = []
    print("  baseline...")
    s, _ = eval_variant("baseline", retrieve_baseline_k, BENCHMARK_QUERIES, judge_fn=cross_judge)
    cross_summaries.append(s)
    for src, variant in loaded_variants.items():
        name = f"graph_{src}_boost"
        print(f"  {name}...")
        fn = lambda q, k, _v=variant: retrieve_graph_rag_with(q, _v, top_n=k)
        s, _ = eval_variant(name, fn, BENCHMARK_QUERIES, judge_fn=cross_judge)
        cross_summaries.append(s)
    print(f"\n🏆 Cross-judge ({CROSS_JUDGE_MODEL}):")
    display(pd.DataFrame(cross_summaries).set_index("variant"))


Evaluando baseline (dense + BM25)...
Evaluando graph_chunks_boost...
Evaluando graph_chunks_ctxonly (sin boost, triples sólo en prompt)...
Evaluando graph_hybrid_boost...
Evaluando graph_hybrid_ctxonly (sin boost, triples sólo en prompt)...
Evaluando graph_original_boost...
Evaluando graph_original_ctxonly (sin boost, triples sólo en prompt)...

🏆 Comparativa global (multi-K):


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
baseline,0.752381,0.899905,0.628571,0.891062,0.900794,0.842912
graph_chunks_boost,0.638095,0.741110,0.542857,0.763292,0.732804,0.708169
graph_chunks_ctxonly,0.761905,0.903521,0.633333,0.895751,0.900794,0.846615
graph_hybrid_boost,0.695238,0.773630,0.623810,0.797311,0.754762,0.748581
graph_hybrid_ctxonly,0.761905,0.901423,0.633333,0.895085,0.900794,0.852926
graph_original_boost,0.704762,0.812927,0.595238,0.810681,0.766667,0.747288
graph_original_ctxonly,0.742857,0.873589,0.614286,0.868385,0.869048,0.825226



Δ NDCG@5 por query (boost - baseline):


,query,base_NDCG@5,graph_chunks_boost_NDCG,Δ_chunks_boost,graph_hybrid_boost_NDCG,Δ_hybrid_boost,graph_original_boost_NDCG,Δ_original_boost
0,¿Qué se decidió sobre el talud?,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
1,¿Cuál es el estado del camino provisional?,0.906528,0.906528,0.000000,0.906528,0.000000,0.986352,0.079824
2,¿Qué incidencias AR-29 aparecen?,1.000000,1.000000,0.000000,0.880740,-0.119260,0.867087,-0.132913
3,¿Qué responsable está asignado a las acciones ...,1.000000,1.000000,0.000000,0.000000,-1.000000,0.000000,-1.000000
4,¿Qué se acordó sobre hormigonado de zapatas?,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
5,Estado de las instalaciones de megafonía,0.864205,0.000000,-0.864205,0.457778,-0.406427,0.734493,-0.129712
6,¿Qué documentación debe aportar la UTE sobre e...,1.000000,1.000000,0.000000,0.618289,-0.381711,0.712263,-0.287737
7,¿Cuál es la función de la sigla DEO en un acta...,1.000000,1.000000,0.000000,0.888722,-0.111278,0.888722,-0.111278
8,¿Cuántos días hay de plazo para devolver el ac...,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
9,¿Qué modelo de luminaria debe instalarse en lo...,1.000000,1.000000,0.000000,0.938300,-0.061700,0.971409,-0.028591



📊 Ganador por métrica (global):
      Precision@5 → graph_chunks_ctxonly (0.762)
           NDCG@5 → graph_chunks_ctxonly (0.904)
     Precision@10 → graph_chunks_ctxonly (0.633)
          NDCG@10 → graph_chunks_ctxonly (0.896)
              MRR → baseline (0.901)
              MAP → graph_hybrid_ctxonly (0.853)

🎯 Subconjunto FACTUAL (lookup específico):


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
baseline,0.916667,0.979663,0.716667,0.967469,1.000000,0.979812
graph_chunks_boost,0.850000,0.907646,0.675000,0.895747,0.916667,0.898163
graph_chunks_ctxonly,0.916667,0.982344,0.716667,0.970150,1.000000,0.979812
graph_hybrid_boost,0.733333,0.773999,0.666667,0.799466,0.751389,0.781796
graph_hybrid_ctxonly,0.916667,0.978672,0.708333,0.967792,1.000000,0.987219
graph_original_boost,0.766667,0.803885,0.633333,0.827278,0.786111,0.798343
graph_original_ctxonly,0.916667,0.968587,0.708333,0.956688,1.000000,0.981497



🌐 Subconjunto TRANSVERSAL (agregadas/analíticas — donde el grafo debe ayudar):


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
baseline,0.533333,0.793561,0.511111,0.789185,0.768519,0.660378
graph_chunks_boost,0.355556,0.519061,0.366667,0.586685,0.487654,0.454843
graph_chunks_ctxonly,0.555556,0.798424,0.522222,0.796552,0.768519,0.669020
graph_hybrid_boost,0.644444,0.773139,0.566667,0.794437,0.759259,0.704295
graph_hybrid_ctxonly,0.555556,0.798424,0.533333,0.798141,0.768519,0.673869
graph_original_boost,0.622222,0.824983,0.544444,0.788552,0.740741,0.679215
graph_original_ctxonly,0.511111,0.746925,0.488889,0.750647,0.694444,0.616866



🧑‍⚖️ Cross-judge con gpt-4o (validación independiente)...
  baseline...
  graph_chunks_boost...
  graph_hybrid_boost...
  graph_original_boost...

🏆 Cross-judge (gpt-4o):


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
baseline,0.533333,0.698117,0.428571,0.753646,0.707672,0.677936
graph_chunks_boost,0.447619,0.564468,0.347619,0.595860,0.554497,0.543282
graph_hybrid_boost,0.419048,0.598073,0.371429,0.629981,0.583466,0.534760
graph_original_boost,0.419048,0.532203,0.342857,0.572982,0.549792,0.501739


## 9. Eval de calidad de RESPUESTA (no sólo retrieval)

Lo que medíamos antes (Precision/NDCG/MAP) puntúa la **relevancia de los chunks recuperados**. Pero el valor real de Graph RAG está en la **respuesta final generada**. Aquí evaluamos eso directamente con un LLM-as-Judge sobre 3 dimensiones por query:

- **Correctness** (0–2): ¿la respuesta es correcta según las fuentes del contexto?
- **Faithfulness** (0–2): ¿se ciñe al contexto sin inventarse hechos?
- **Citation** (0–2): ¿cita correctamente fuente(s) `(Fuente: ...)`?

> Coste: ~0,02 USD por variante × ~21 queries = total ~0,1 USD para las 4 variantes (baseline + 3 grafos boost).

Saltar si `ANSWER_EVAL_ENABLED = False`.


In [16]:
def build_prompt_baseline(query, chunks):
    """Prompt baseline (sin grafo)."""
    blocks = []
    for i, ch in enumerate(chunks, 1):
        doc_id = ch["meta"]["doc_id"]
        text = strip_doc_prefix(ch["text"], EMBED_BACKEND)
        blocks.append(f"[Fragmento {i} - Fuente: {doc_id}]\n{text}")
    return f"""Eres COSORA, un asistente técnico especializado en análisis de actas de obra ferroviaria en España.

REGLAS:
1. Responde ÚNICAMENTE con información del CONTEXTO.
2. Si no está, dilo explícitamente.
3. Cita la fuente entre paréntesis: (Fuente: nombre_del_acta).

VOCABULARIO: DF=Dirección Facultativa, UTE=Unión Temporal de Empresas, DEO=Director de Ejecución de Obra.

=== CONTEXTO ===
{chr(10).join(blocks)}

=== PREGUNTA ===
{query}

=== RESPUESTA ==="""


def generate_answer_for_variant(query, variant_name, variant=None, top_n=TOP_N):
    """Devuelve (answer, chunks, triples) para una variante dada.

    Sufijos soportados:
      - _boost          → grafo en RRF (reranking) + triples en prompt (W2: strict)
      - _boost_noprompt → grafo en RRF (reranking) PERO triples NO van al prompt (W1)
      - _ctxonly        → ranking igual al baseline, triples sólo en prompt (W2: strict)
    """
    if variant_name == "baseline":
        chunks = retrieve_baseline(query, top_n=top_n)
        triples = []
        prompt = build_prompt_baseline(query, chunks)
    elif variant_name.endswith("_boost_noprompt"):
        # W1: el grafo SÍ reordena el ranking, pero el LLM no ve triples
        chunks = retrieve_graph_rag_with(query, variant, top_n=top_n)
        triples = retrieve_graph_variant(query, variant, k=GRAPH_TOP_K)  # retornado solo para inspección
        prompt = build_prompt_baseline(query, chunks)  # prompt sin triples
    elif variant_name.endswith("_boost"):
        chunks = retrieve_graph_rag_with(query, variant, top_n=top_n)
        triples = retrieve_graph_variant(query, variant, k=GRAPH_TOP_K)
        prompt = build_prompt_graph(query, chunks, triples)
    elif variant_name.endswith("_ctxonly"):
        chunks = retrieve_baseline(query, top_n=top_n)
        triples = retrieve_graph_variant(query, variant, k=GRAPH_TOP_K)
        prompt = build_prompt_graph(query, chunks, triples)
    else:
        raise ValueError(f"Variante desconocida: {variant_name}")
    return generate_answer(prompt), chunks, triples


print("✅ Funciones de generación por variante listas")


✅ Funciones de generación por variante listas


In [17]:
def judge_answer(query: str, answer: str, context_blocks: list[str], model: str = "gpt-4o-mini") -> dict:
    """LLM-as-Judge sobre la RESPUESTA en 3 ejes (correctness, faithfulness, citation).

    Devuelve dict con scores 0/1/2 o -1 si no parseable.
    """
    ctx = "\n\n---\n\n".join(context_blocks[:8])  # cap para no inflar prompt
    system = (
        "Eres un evaluador estricto de respuestas de un asistente sobre actas de obra ferroviaria. "
        "Vas a puntuar la RESPUESTA en tres ejes (0/1/2). "
        "Responde EXACTAMENTE en formato JSON: {\"correctness\":X,\"faithfulness\":Y,\"citation\":Z}."
    )
    user = (
        f"PREGUNTA:\n{query}\n\n"
        f"CONTEXTO PROVISTO AL ASISTENTE:\n{ctx}\n\n"
        f"RESPUESTA DEL ASISTENTE:\n{answer}\n\n"
        "Puntúa (0/1/2):\n"
        "- correctness: ¿la respuesta es correcta a la luz del contexto?\n"
        "- faithfulness: ¿se ciñe al contexto, sin inventar hechos no soportados?\n"
        "- citation: ¿cita fuentes (Fuente: ...) y las cita bien?\n"
        "Devuelve SOLO el JSON."
    )
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0, max_tokens=60,
        )
        raw = (resp.choices[0].message.content or "").strip()
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if not m:
            return {"correctness": -1, "faithfulness": -1, "citation": -1, "raw": raw}
        data = json.loads(m.group(0))
        return {
            "correctness": int(data.get("correctness", -1)),
            "faithfulness": int(data.get("faithfulness", -1)),
            "citation": int(data.get("citation", -1)),
        }
    except Exception as e:
        return {"correctness": -1, "faithfulness": -1, "citation": -1, "error": str(e)}


def eval_answer_variant(variant_name, variant_obj, queries):
    rows = []
    for q in queries:
        ans, chunks, triples = generate_answer_for_variant(q, variant_name, variant=variant_obj)
        ctx_blocks = [strip_doc_prefix(c["text"], EMBED_BACKEND) for c in chunks]
        if triples:
            ctx_blocks.append("[GRAFO]\n" + "\n".join(t["text"] for t in triples))
        scores = judge_answer(q, ans, ctx_blocks)
        rows.append({"query": q[:60], "answer": ans, **scores})
    return rows


def summarize_answer_rows(name, rows):
    def avg(field):
        vals = [r[field] for r in rows if r.get(field, -1) >= 0]
        return float(np.mean(vals)) if vals else 0.0
    return {
        "variant": name,
        "correctness": avg("correctness"),
        "faithfulness": avg("faithfulness"),
        "citation": avg("citation"),
        "overall": avg("correctness") * 0.5 + avg("faithfulness") * 0.35 + avg("citation") * 0.15,
    }


print("✅ Juez de respuesta listo (correctness/faithfulness/citation)")


✅ Juez de respuesta listo (correctness/faithfulness/citation)


In [18]:
if ANSWER_EVAL_ENABLED:
    answer_summaries = []
    answer_details = {}

    print("Evaluando RESPUESTAS · baseline...")
    rows = eval_answer_variant("baseline", None, BENCHMARK_QUERIES)
    answer_summaries.append(summarize_answer_rows("baseline", rows))
    answer_details["baseline"] = rows

    for src, variant in loaded_variants.items():
        # W1+W2: incluimos _boost_noprompt para aislar el efecto del prompt vs el ranking
        for suffix in ("_boost", "_boost_noprompt", "_ctxonly"):
            name = f"graph_{src}{suffix}"
            print(f"Evaluando RESPUESTAS · {name}...")
            rows = eval_answer_variant(name, variant, BENCHMARK_QUERIES)
            answer_summaries.append(summarize_answer_rows(name, rows))
            answer_details[name] = rows

    df_ans = pd.DataFrame(answer_summaries).set_index("variant")
    print("\n🏆 Calidad de RESPUESTA (no sólo retrieval):")
    display(df_ans.sort_values("overall", ascending=False))

    # Subset transversal — donde el grafo debe ayudar más
    print("\n🌐 Subconjunto TRANSVERSAL (calidad de respuesta):")
    rows_t = []
    for name, all_rows in answer_details.items():
        sub = [r for r, q in zip(all_rows, BENCHMARK_QUERIES) if q in BENCHMARK_QUERIES_TRANSVERSAL]
        rows_t.append(summarize_answer_rows(name, sub))
    display(pd.DataFrame(rows_t).set_index("variant").sort_values("overall", ascending=False))
else:
    print("ANSWER_EVAL_ENABLED=False → saltado")


Evaluando RESPUESTAS · baseline...
Evaluando RESPUESTAS · graph_chunks_boost...
Evaluando RESPUESTAS · graph_chunks_boost_noprompt...
Evaluando RESPUESTAS · graph_chunks_ctxonly...
Evaluando RESPUESTAS · graph_hybrid_boost...
Evaluando RESPUESTAS · graph_hybrid_boost_noprompt...

🏆 Calidad de RESPUESTA (no sólo retrieval):


,correctness,faithfulness,citation,overall
variant,,,,
baseline,1.714286,1.904762,1.476190,1.745238
graph_chunks_boost,1.761905,1.952381,1.047619,1.721429
graph_chunks_ctxonly,1.714286,1.952381,1.000000,1.690476
graph_chunks_boost_noprompt,1.666667,1.904762,1.142857,1.671429
graph_hybrid_boost_noprompt,1.619048,1.952381,1.047619,1.650000
graph_original_boost_noprompt,1.619048,1.904762,1.142857,1.647619
graph_hybrid_ctxonly,1.571429,1.809524,1.142857,1.590476
graph_hybrid_boost,1.571429,1.857143,1.000000,1.585714
graph_original_ctxonly,1.523810,1.809524,1.000000,1.545238



🌐 Subconjunto TRANSVERSAL (calidad de respuesta):


,correctness,faithfulness,citation,overall
variant,,,,
graph_chunks_boost,1.666667,2.000000,0.888889,1.666667
graph_chunks_boost_noprompt,1.444444,1.888889,0.888889,1.516667
graph_chunks_ctxonly,1.444444,1.888889,0.888889,1.516667
baseline,1.333333,1.777778,1.222222,1.472222
graph_hybrid_boost_noprompt,1.333333,1.888889,0.888889,1.461111
graph_original_boost_noprompt,1.222222,1.777778,1.000000,1.383333
graph_hybrid_boost,1.222222,1.666667,0.888889,1.327778
graph_hybrid_ctxonly,1.000000,1.555556,0.888889,1.177778
graph_original_boost,1.000000,1.555556,0.777778,1.161111


## 10. Debug — ¿por qué el grafo empeora ciertas queries?

Caso de estudio típico: **"megafonía"**. El grafo empuja chunks aparentemente irrelevantes, degradando NDCG. Aquí inspeccionamos qué triples top-K aparecen y qué chunks reciben boost, para cada variante de grafo.


In [19]:
DEBUG_QUERY = "Estado de las instalaciones de megafonía"

print(f"🔎 Debug query: {DEBUG_QUERY!r}\n")

# 1) Baseline: qué recupera SIN grafo
base_chunks = retrieve_baseline(DEBUG_QUERY, top_n=EVAL_K)
print("─── BASELINE (dense + BM25) ───")
for c in base_chunks:
    txt = strip_doc_prefix(c["text"], EMBED_BACKEND)[:120].replace("\n", " ")
    print(f"  [{c['meta']['doc_id']}] score={c['score']:.4f} | {txt}…")

# 2) Por cada variante: triples top-K y chunks boosteados
for src, variant in loaded_variants.items():
    print(f"\n─── GRAPH variant: {src} ───")
    triples = retrieve_graph_variant(DEBUG_QUERY, variant, k=GRAPH_TOP_K)
    print(f"Triples top-{len(triples)} (cos):")
    for t in triples:
        flag = "✓boost" if t["score"] >= GRAPH_MIN_COS else " skip "
        print(f"  [{flag}] cos={t['score']:.3f} | {t['text']}")

    boosted_chunks = retrieve_graph_rag_with(DEBUG_QUERY, variant, top_n=EVAL_K)
    print(f"\nChunks tras RRF+boost (top-{EVAL_K}):")
    for c in boosted_chunks:
        boost = c.get("graph_boost", 0)
        txt = strip_doc_prefix(c["text"], EMBED_BACKEND)[:120].replace("\n", " ")
        tag = f" g×{boost}" if boost else ""
        print(f"  [{c['meta']['doc_id']}] score={c['score']:.4f}{tag} | {txt}…")

    # ¿Qué chunks NUEVOS (no estaban en baseline top-K) introdujo el grafo?
    base_ids = {(c["meta"]["doc_id"], c["text"][:40]) for c in base_chunks}
    new_in_graph = [c for c in boosted_chunks
                    if (c["meta"]["doc_id"], c["text"][:40]) not in base_ids]
    if new_in_graph:
        print(f"\n⚠️  El grafo introdujo {len(new_in_graph)} chunk(s) que el baseline no tenía:")
        for c in new_in_graph:
            txt = strip_doc_prefix(c["text"], EMBED_BACKEND)[:200].replace("\n", " ")
            print(f"  [{c['meta']['doc_id']}] g×{c.get('graph_boost', 0)}\n    {txt}…")


🔎 Debug query: 'Estado de las instalaciones de megafonía'

─── BASELINE (dense + BM25) ───
  [254275-DO-AVO-21-V01] score=0.0331 | d: 1,70m), y las dimensiones de las placas de anclaje tipo 2 correspondientes a los soportes P-02 y P-06 (33x25 cm). Det…
  [244170-DOB-AVO-19-V00-A0] score=0.0328 | existente, donde se ha instalado el nuevo cableado para la instalación de megafonía. [pic] Fotografía 05. Unidad climati…
  [254275-DO-AVO-21-V01] score=0.0320 | teleindicador || Tras consultar con DC, se confirma que este cableado ha de ser ejecutado por la UTE, y se informa a la …
  [244170-DOB-AVO-19-V00-A0] score=0.0317 | Convocatoria próxima reunión | | | |Fecha: | |Lugar: | ANEJO 1. FOTOGRAFÍAS [pic] Fotografía 01. Nuevo Falso techo del v…
  [254275-DO-AVO-14-V07] score=0.0299 | se estudiará su viabilidad. Alzado. Detalle. Estado reformado. Como información adicional a esta acta, se indica que el …
  [254275-DO-AVO-17-V07] score=0.0294 | os de los ensayos de compactación, y que los enviar

## 11. Router de tipo de query — `graph_router`

**Hipótesis (Sesión B / M1):** las queries factuales no necesitan grafo (baseline ya ≈techo, NDCG@5=0.99). Las transversales sí ganan con `*_boost_noprompt` (+5–18% según métrica). Un **router** que clasifica la query antes de retrieval y elige pipeline debería combinar lo mejor de ambos.

**Diseño en 2 niveles:**
1. **Trigger words** (regex): rápido y gratis. Si la query contiene "todas/agrupa/compara/frecuentes/tipo/lista/patrón/conjunto/varias..." → transversal.
2. **Fallback LLM** (`gpt-4o-mini` con few-shot): solo si nivel 1 no decide con confianza. ~100ms, ~$0.0001 por query.

**Variante propuesta:** `graph_router` = baseline si factual, `graph_original_boost_noprompt` si transversal (la que ganó en transversales).


In [20]:
# ─── Clasificador factual / transversal ─────────────────────────────

# Palabras que fuertemente indican una query analítica/agregada
TRANSVERSAL_TRIGGERS = [
    "todas", "todos", "todas las", "todos los",
    "agrupa", "agrupar", "clasifica", "clasificar",
    "compara", "comparar", "comparación",
    "frecuente", "frecuentes", "frecuencia",
    "lista", "listar", "enumera", "enumerar",
    "tipos de", "tipo de tareas", "qué tipo",
    "patrón", "patrones", "tendencia", "tendencias",
    "conjunto", "varias actas", "todas las actas",
    "general", "globalmente", "en general",
    "qué incidencias", "qué problemas", "qué elementos",
    "más comunes", "más frecuentes",
    "cuáles son las",
    "incumplimientos", "solicitudes realizadas",
]

# Palabras que indican factual / lookup específico
FACTUAL_TRIGGERS = [
    "qué se decidió", "qué se acordó",
    "estado del", "estado de la",
    "cuál es", "cuáles es",
    "cuántos", "cuántas",
    "a qué altura", "a qué distancia",
    "modelo de", "función de", "sigla",
    "documentación", "certificados",
    "ar-29",
]


def _normalize_q(q: str) -> str:
    return q.lower().strip()


def classify_query_regex(query: str) -> str | None:
    """Clasificación rápida por palabras-trigger. Devuelve 'factual',
    'transversal' o None si no decide."""
    q = _normalize_q(query)
    score_t = sum(1 for w in TRANSVERSAL_TRIGGERS if w in q)
    score_f = sum(1 for w in FACTUAL_TRIGGERS if w in q)
    if score_t >= 1 and score_t > score_f:
        return "transversal"
    if score_f >= 1 and score_f > score_t:
        return "factual"
    return None


_CLASSIFY_SYSTEM = (
    "Eres un clasificador binario de consultas sobre actas de obra ferroviaria. "
    "Tu única tarea: emitir una etiqueta entre 'factual' y 'transversal'.\n"
    "- factual: pregunta puntual sobre un dato concreto contenido en UN documento "
    "(p.ej. '¿Qué modelo de luminaria?', 'Estado del camino', '¿Cuántos días para devolver?').\n"
    "- transversal: pregunta agregada que requiere sintetizar VARIOS documentos "
    "(p.ej. '¿Cuáles son las incidencias más frecuentes?', 'Lista todas las solicitudes', "
    "'Compara obras')."
)

_CLASSIFY_FEW_SHOT = [
    ("¿Qué se decidió sobre el talud?", "factual"),
    ("¿Cuántos días hay para devolver el acta?", "factual"),
    ("¿Cuáles son las incidencias más frecuentes en todas las actas?", "transversal"),
    ("Lista todas las solicitudes a la constructora", "transversal"),
    ("¿Qué modelo de luminaria debe instalarse?", "factual"),
    ("Agrupa las incidencias por tipo", "transversal"),
]


def classify_query_llm(query: str) -> str:
    """Fallback con LLM. Más caro (~$0.0001) pero entiende contexto."""
    msgs = [{"role": "system", "content": _CLASSIFY_SYSTEM}]
    for q, lbl in _CLASSIFY_FEW_SHOT:
        msgs.append({"role": "user", "content": q})
        msgs.append({"role": "assistant", "content": lbl})
    msgs.append({"role": "user", "content": query})
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=msgs,
            temperature=0,
            max_tokens=4,
        )
        out = (resp.choices[0].message.content or "").strip().lower()
        if "transversal" in out:
            return "transversal"
        return "factual"  # default conservador
    except Exception:
        return "factual"  # ante duda, baseline


def classify_query(query: str) -> str:
    """Clasificación en 2 niveles: regex → LLM fallback."""
    label = classify_query_regex(query)
    if label is not None:
        return label
    return classify_query_llm(query)


# Sanity check sobre el benchmark
print("🧪 Clasificación del benchmark:")
correct = 0
for q in BENCHMARK_QUERIES:
    pred = classify_query(q)
    truth = "factual" if q in BENCHMARK_QUERIES_FACTUAL else "transversal"
    ok = "✓" if pred == truth else "✗"
    if pred == truth:
        correct += 1
    print(f"  {ok} [pred={pred:>11} | truth={truth:>11}] {q[:70]}")
print(f"\nPrecisión del router: {correct}/{len(BENCHMARK_QUERIES)} = {correct/len(BENCHMARK_QUERIES):.1%}")


🧪 Clasificación del benchmark:
  ✓ [pred=    factual | truth=    factual] ¿Qué se decidió sobre el talud?
  ✓ [pred=    factual | truth=    factual] ¿Cuál es el estado del camino provisional?
  ✓ [pred=    factual | truth=    factual] ¿Qué incidencias AR-29 aparecen?
  ✓ [pred=    factual | truth=    factual] ¿Qué responsable está asignado a las acciones sobre el talud?
  ✓ [pred=    factual | truth=    factual] ¿Qué se acordó sobre hormigonado de zapatas?
  ✓ [pred=    factual | truth=    factual] Estado de las instalaciones de megafonía
  ✓ [pred=    factual | truth=    factual] ¿Qué documentación debe aportar la UTE sobre estabilidad?
  ✓ [pred=    factual | truth=    factual] ¿Cuál es la función de la sigla DEO en un acta de visita de obra?
  ✓ [pred=    factual | truth=    factual] ¿Cuántos días hay de plazo para devolver el acta firmada?
  ✓ [pred=    factual | truth=    factual] ¿Qué modelo de luminaria debe instalarse en los báculos del andén?
  ✓ [pred=    factual | truth=    

In [21]:
# ─── Variante graph_router ───────────────────────────────────────────
# Elige el grafo a usar en transversales. Por defecto el ganador de la sec 9:
ROUTER_TRANSVERSAL_SOURCE = "original"   # "chunks" | "hybrid" | "original"

_router_variant = loaded_variants.get(ROUTER_TRANSVERSAL_SOURCE)
assert _router_variant is not None, f"Grafo '{ROUTER_TRANSVERSAL_SOURCE}' no cargado"


def retrieve_router(query, top_n=TOP_N):
    """Retrieval con router: factual→baseline, transversal→graph boost (sin triples al prompt)."""
    label = classify_query(query)
    if label == "transversal":
        return retrieve_graph_rag_with(query, _router_variant, top_n=top_n)
    return retrieve_baseline(query, top_n=top_n)


def retrieve_router_k(query, k):
    return retrieve_router(query, top_n=k)


def generate_router_answer(query, top_n=TOP_N):
    """Generación con router: pipeline completo (retrieval + prompt) según tipo de query.
    Para transversales usa boost_noprompt (sin triples en el prompt, igual al ganador
    de la sec 9). Para factuales usa el prompt baseline.
    """
    label = classify_query(query)
    if label == "transversal":
        chunks = retrieve_graph_rag_with(query, _router_variant, top_n=top_n)
        triples = retrieve_graph_variant(query, _router_variant, k=GRAPH_TOP_K)
        prompt = build_prompt_baseline(query, chunks)  # SIN triples en prompt (W1)
        return generate_answer(prompt), chunks, triples, label
    chunks = retrieve_baseline(query, top_n=top_n)
    prompt = build_prompt_baseline(query, chunks)
    return generate_answer(prompt), chunks, [], label


# Test rápido
print("🧪 Ejemplo router con 4 queries:")
for q in [
    "¿Qué se decidió sobre el talud?",
    "¿Cuáles son las incidencias más frecuentes?",
    "Estado de las instalaciones de megafonía",
    "Lista todas las solicitudes a la constructora",
]:
    label = classify_query(q)
    print(f"  [{label:>11}] {q}")


🧪 Ejemplo router con 4 queries:
  [    factual] ¿Qué se decidió sobre el talud?
  [transversal] ¿Cuáles son las incidencias más frecuentes?
  [    factual] Estado de las instalaciones de megafonía
  [transversal] Lista todas las solicitudes a la constructora


In [22]:
# ─── Evaluación del router ───────────────────────────────────────────
# 1) Retrieval: precision/NDCG/MRR/MAP @ K=5/10 — comparado con baseline y mejores grafos
# 2) Respuesta: correctness/faithfulness/citation

# 1) RETRIEVAL
print("📏 Evaluando retrieval · graph_router...")
router_ret_summary, router_ret_detail = eval_variant("graph_router", retrieve_router_k, BENCHMARK_QUERIES)

# Referencia: ya tenemos en memoria los summaries previos (all_summaries de sec 8)
try:
    df_router_ret = pd.DataFrame(all_summaries + [router_ret_summary]).set_index("variant")
except NameError:
    df_router_ret = pd.DataFrame([router_ret_summary]).set_index("variant")
    print("⚠️  Ejecuta sec 8 antes para tener referencias")

print("\n🏆 Retrieval — graph_router vs todas las variantes:")
display(df_router_ret.sort_values("NDCG@5", ascending=False))

# Por subconjunto
print("\n🎯 Retrieval FACTUAL:")
rows_f = [_summary_subset(n, all_details[n], BENCHMARK_QUERIES_FACTUAL) for n in all_details]
rows_f.append(_summary_subset("graph_router", router_ret_detail, BENCHMARK_QUERIES_FACTUAL))
display(pd.DataFrame([r for r in rows_f if r]).set_index("variant").sort_values("NDCG@5", ascending=False))

print("\n🌐 Retrieval TRANSVERSAL:")
rows_t = [_summary_subset(n, all_details[n], BENCHMARK_QUERIES_TRANSVERSAL) for n in all_details]
rows_t.append(_summary_subset("graph_router", router_ret_detail, BENCHMARK_QUERIES_TRANSVERSAL))
display(pd.DataFrame([r for r in rows_t if r]).set_index("variant").sort_values("NDCG@5", ascending=False))


📏 Evaluando retrieval · graph_router...

🏆 Retrieval — graph_router vs todas las variantes:


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
graph_router,0.780952,0.921477,0.638095,0.897336,0.888889,0.856222
graph_chunks_ctxonly,0.761905,0.903521,0.633333,0.895751,0.900794,0.846615
graph_hybrid_ctxonly,0.761905,0.901423,0.633333,0.895085,0.900794,0.852926
baseline,0.752381,0.899905,0.628571,0.891062,0.900794,0.842912
graph_original_ctxonly,0.742857,0.873589,0.614286,0.868385,0.869048,0.825226
graph_original_boost,0.704762,0.812927,0.595238,0.810681,0.766667,0.747288
graph_hybrid_boost,0.695238,0.773630,0.623810,0.797311,0.754762,0.748581
graph_chunks_boost,0.638095,0.741110,0.542857,0.763292,0.732804,0.708169



🎯 Retrieval FACTUAL:


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
graph_router,0.916667,0.990980,0.716667,0.978785,1.000000,0.979812
graph_chunks_ctxonly,0.916667,0.982344,0.716667,0.970150,1.000000,0.979812
baseline,0.916667,0.979663,0.716667,0.967469,1.000000,0.979812
graph_hybrid_ctxonly,0.916667,0.978672,0.708333,0.967792,1.000000,0.987219
graph_original_ctxonly,0.916667,0.968587,0.708333,0.956688,1.000000,0.981497
graph_chunks_boost,0.850000,0.907646,0.675000,0.895747,0.916667,0.898163
graph_original_boost,0.766667,0.803885,0.633333,0.827278,0.786111,0.798343
graph_hybrid_boost,0.733333,0.773999,0.666667,0.799466,0.751389,0.781796



🌐 Retrieval TRANSVERSAL:


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
graph_router,0.600000,0.828806,0.533333,0.788736,0.740741,0.691434
graph_original_boost,0.622222,0.824983,0.544444,0.788552,0.740741,0.679215
graph_chunks_ctxonly,0.555556,0.798424,0.522222,0.796552,0.768519,0.669020
graph_hybrid_ctxonly,0.555556,0.798424,0.533333,0.798141,0.768519,0.673869
baseline,0.533333,0.793561,0.511111,0.789185,0.768519,0.660378
graph_hybrid_boost,0.644444,0.773139,0.566667,0.794437,0.759259,0.704295
graph_original_ctxonly,0.511111,0.746925,0.488889,0.750647,0.694444,0.616866
graph_chunks_boost,0.355556,0.519061,0.366667,0.586685,0.487654,0.454843


In [23]:
# 2) CALIDAD DE RESPUESTA con el router
if ANSWER_EVAL_ENABLED:
    print("📏 Evaluando RESPUESTAS · graph_router...")
    rows = []
    label_dist = {"factual": 0, "transversal": 0}
    for q in BENCHMARK_QUERIES:
        ans, chunks, triples, label = generate_router_answer(q)
        label_dist[label] += 1
        ctx_blocks = [strip_doc_prefix(c["text"], EMBED_BACKEND) for c in chunks]
        scores = judge_answer(q, ans, ctx_blocks)
        rows.append({"query": q[:60], "label": label, "answer": ans, **scores})

    router_ans_summary = summarize_answer_rows("graph_router", rows)
    print(f"\nDistribución de etiquetas del router: {label_dist}")

    # Tabla global comparada con todas las variantes de respuesta
    try:
        df_all_ans = pd.DataFrame(answer_summaries + [router_ans_summary]).set_index("variant")
    except NameError:
        df_all_ans = pd.DataFrame([router_ans_summary]).set_index("variant")
        print("⚠️  Ejecuta sec 9 antes para tener referencias")

    print("\n🏆 Calidad de RESPUESTA — graph_router vs resto:")
    display(df_all_ans.sort_values("overall", ascending=False))

    # Subset transversal
    print("\n🌐 RESPUESTA — Subset TRANSVERSAL:")
    rows_t = []
    for name, all_rows in answer_details.items():
        sub = [r for r, q in zip(all_rows, BENCHMARK_QUERIES) if q in BENCHMARK_QUERIES_TRANSVERSAL]
        rows_t.append(summarize_answer_rows(name, sub))
    sub_router = [r for r, q in zip(rows, BENCHMARK_QUERIES) if q in BENCHMARK_QUERIES_TRANSVERSAL]
    rows_t.append(summarize_answer_rows("graph_router", sub_router))
    display(pd.DataFrame(rows_t).set_index("variant").sort_values("overall", ascending=False))

    # Diagnóstico: dónde se equivoca el router
    print("\n🔍 Decisiones del router por query:")
    for q, r in zip(BENCHMARK_QUERIES, rows):
        truth = "factual" if q in BENCHMARK_QUERIES_FACTUAL else "transversal"
        ok = "✓" if r["label"] == truth else "✗"
        print(f"  {ok} [pred={r['label']:>11} | truth={truth:>11}] overall={r['correctness']*0.5 + r['faithfulness']*0.35 + r['citation']*0.15:.2f} | {q[:60]}")
else:
    print("ANSWER_EVAL_ENABLED=False → saltado")


📏 Evaluando RESPUESTAS · graph_router...

Distribución de etiquetas del router: {'factual': 12, 'transversal': 9}

🏆 Calidad de RESPUESTA — graph_router vs resto:


,correctness,faithfulness,citation,overall
variant,,,,
baseline,1.714286,1.904762,1.476190,1.745238
graph_chunks_boost,1.761905,1.952381,1.047619,1.721429
graph_router,1.714286,1.952381,1.190476,1.719048
graph_chunks_ctxonly,1.714286,1.952381,1.000000,1.690476
graph_chunks_boost_noprompt,1.666667,1.904762,1.142857,1.671429
graph_hybrid_boost_noprompt,1.619048,1.952381,1.047619,1.650000
graph_original_boost_noprompt,1.619048,1.904762,1.142857,1.647619
graph_hybrid_ctxonly,1.571429,1.809524,1.142857,1.590476
graph_hybrid_boost,1.571429,1.857143,1.000000,1.585714



🌐 RESPUESTA — Subset TRANSVERSAL:


,correctness,faithfulness,citation,overall
variant,,,,
graph_chunks_boost,1.666667,2.000000,0.888889,1.666667
graph_chunks_boost_noprompt,1.444444,1.888889,0.888889,1.516667
graph_chunks_ctxonly,1.444444,1.888889,0.888889,1.516667
graph_router,1.444444,1.888889,0.888889,1.516667
baseline,1.333333,1.777778,1.222222,1.472222
graph_hybrid_boost_noprompt,1.333333,1.888889,0.888889,1.461111
graph_original_boost_noprompt,1.222222,1.777778,1.000000,1.383333
graph_hybrid_boost,1.222222,1.666667,0.888889,1.327778
graph_hybrid_ctxonly,1.000000,1.555556,0.888889,1.177778



🔍 Decisiones del router por query:
  ✓ [pred=    factual | truth=    factual] overall=1.85 | ¿Qué se decidió sobre el talud?
  ✓ [pred=    factual | truth=    factual] overall=2.00 | ¿Cuál es el estado del camino provisional?
  ✓ [pred=    factual | truth=    factual] overall=1.85 | ¿Qué incidencias AR-29 aparecen?
  ✓ [pred=    factual | truth=    factual] overall=2.00 | ¿Qué responsable está asignado a las acciones sobre el talud
  ✓ [pred=    factual | truth=    factual] overall=1.20 | ¿Qué se acordó sobre hormigonado de zapatas?
  ✓ [pred=    factual | truth=    factual] overall=2.00 | Estado de las instalaciones de megafonía
  ✓ [pred=    factual | truth=    factual] overall=1.85 | ¿Qué documentación debe aportar la UTE sobre estabilidad?
  ✓ [pred=    factual | truth=    factual] overall=1.85 | ¿Cuál es la función de la sigla DEO en un acta de visita de 
  ✓ [pred=    factual | truth=    factual] overall=1.85 | ¿Cuántos días hay de plazo para devolver el acta firmada?
  ✓ [pred=

## 12. M3 — Entity linking explícito

**Problema:** el retrieval por coseno sobre triples completos mete ruido cuando la query menciona una entidad que **no está** en el grafo (ej. "megafonía" → triples sobre "estructura metálica" con cos≈0.82).

**Solución M3:**
1. Indexar **entidades** únicas del grafo (sujetos/objetos de los triples).
2. Extraer entidades de la query con LLM (dominio ferroviario: UTE, DEO, AR-29, talud…).
3. Resolver query-entidad → entidad-grafo por coseno (`ENTITY_MIN_COS`).
4. Si **ningún match** → grafo desactivado (equivale a baseline en el boost).
5. Si hay match → solo triples que **mencionan** esas entidades, rankeados por coseno con la query.

**Variantes nuevas:**
- `graph_entity_boost_noprompt` — entity linking + boost RRF, sin triples en prompt (W1).
- `graph_router_entity` — router factual/transversal + entity linking en transversales.


In [25]:
# ─── Entity index + extracción + resolución ──────────────────────────

_entity_extract_cache: dict[str, list[str]] = {}


def build_entity_index(variant: dict) -> dict:
    """Añade a `variant` el índice de entidades únicas (sujetos/objetos)."""
    entities = set()
    for s, _, o in variant["relations"]:
        if s and s.strip():
            entities.add(s.strip())
        if o and o.strip():
            entities.add(o.strip())
    entities = sorted(entities)
    variant["entities"] = entities
    if not entities:
        variant["entity_norm"] = None
        return variant
    vecs = embedder.embed_batch(entities, is_query=False, batch_size=32)
    mat = np.array(vecs, dtype=np.float32)
    variant["entity_norm"] = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)
    return variant


def _triple_mentions_entity(triple, graph_entity: str) -> bool:
    """True si el sujeto u objeto del triple menciona la entidad del grafo."""
    s, _, o = triple
    ge = graph_entity.lower()
    sl, ol = s.lower(), o.lower()
    return ge in sl or ge in ol or sl in ge or ol in ge


_EXTRACT_SYSTEM = (
    "Extraes entidades técnicas de consultas sobre actas de obra ferroviaria en España.\n"
    "INCLUIR: elementos físicos (talud, megafonía, luminaria, zapatas), actores (UTE, DEO, DF, RENFE), "
    "códigos (AR-29, 244170), instalaciones concretas.\n"
    "EXCLUIR: verbos, palabras de pregunta, conceptos vagos (estado, situación, problema, incidencia genérica "
    "sin objeto concreto).\n"
    "Responde SOLO JSON: {\"entities\": [\"...\", ...]}"
)

_EXTRACT_FEW_SHOT = [
    ("¿Qué se decidió sobre el talud?", '["talud"]'),
    ("Estado de las instalaciones de megafonía", '["megafonía", "instalaciones de megafonía"]'),
    ("¿Qué incidencias AR-29 aparecen?", '["AR-29", "incidencias AR-29"]'),
    ("¿Cuáles son las incidencias más frecuentes en todas las actas?", '[]'),
    ("Lista todas las solicitudes realizadas a la constructora", '["constructora", "UTE"]'),
]


def extract_query_entities(query: str, model: str = ENTITY_EXTRACT_MODEL) -> list[str]:
    """Extrae entidades de la query. Cache por query para no repetir LLM en eval."""
    if query in _entity_extract_cache:
        return _entity_extract_cache[query]
    msgs = [{"role": "system", "content": _EXTRACT_SYSTEM}]
    for q, ans in _EXTRACT_FEW_SHOT:
        msgs.append({"role": "user", "content": q})
        msgs.append({"role": "assistant", "content": ans})
    msgs.append({"role": "user", "content": query})
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=msgs,
            temperature=0,
            max_tokens=80,
        )
        raw = (resp.choices[0].message.content or "").strip()
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        data = json.loads(m.group(0)) if m else {"entities": []}
        entities = [e.strip() for e in data.get("entities", []) if e and str(e).strip()]
    except Exception:
        entities = []
    _entity_extract_cache[query] = entities
    return entities


def resolve_entities_to_graph(query_entities: list[str], variant: dict, min_cos: float = ENTITY_MIN_COS) -> list[str]:
    """Resuelve entidades de la query a entidades del grafo por coseno."""
    if not query_entities or not variant.get("entities") or variant.get("entity_norm") is None:
        return []
    matched = []
    for qe in query_entities:
        qv = np.array(embedder.embed_one(qe, is_query=True), dtype=np.float32)
        qv = qv / (np.linalg.norm(qv) + 1e-12)
        sims = variant["entity_norm"] @ qv
        best_i = int(np.argmax(sims))
        if float(sims[best_i]) >= min_cos:
            ge = variant["entities"][best_i]
            if ge not in matched:
                matched.append(ge)
    return matched


def retrieve_graph_entity(query: str, variant: dict, k: int = GRAPH_TOP_K, min_entity_cos: float = ENTITY_MIN_COS):
    """Triples filtrados por entity linking. [] si no hay entidades resueltas → sin boost."""
    q_ents = extract_query_entities(query)
    matched = resolve_entities_to_graph(q_ents, variant, min_cos=min_entity_cos)

    debug = {"query_entities": q_ents, "matched_graph_entities": matched}

    if not matched:
        return [], debug

    cand_idxs = []
    for i, rel in enumerate(variant["relations"]):
        if any(_triple_mentions_entity(rel, ent) for ent in matched):
            cand_idxs.append(i)

    if not cand_idxs:
        return [], debug

    qv = np.array(embedder.embed_one(query, is_query=True), dtype=np.float32)
    qv = qv / (np.linalg.norm(qv) + 1e-12)
    sims = variant["norm"][cand_idxs] @ qv
    order = np.argsort(-sims)[:k]
    triples = []
    for rk, j in enumerate(order):
        i = cand_idxs[j]
        triples.append({
            "triple": variant["relations"][i],
            "text": variant["texts"][i],
            "score": float(sims[j]),
            "rank_graph": rk,
        })
    return triples, debug


def retrieve_graph_rag_entity(query, variant, top_n=TOP_N):
    """RRF con triples filtrados por entity linking."""
    d = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    b = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    g, _ = retrieve_graph_entity(query, variant, k=GRAPH_TOP_K)
    return rrf_fusion_graph(d, b, g, all_docs, all_metas, rrf_k=RRF_K, top_n=top_n)


def retrieve_graph_entity_k(query, k):
    return retrieve_graph_rag_entity(query, _entity_variant, top_n=k)


# Enriquecer variantes cargadas con índice de entidades
for _src, _v in loaded_variants.items():
    build_entity_index(_v)
    print(f"[{_src:>8}] {len(_v['entities'])} entidades indexadas")

_entity_variant = loaded_variants.get(ENTITY_SOURCE)
assert _entity_variant is not None, f"Grafo '{ENTITY_SOURCE}' no cargado para entity linking"

print(f"\n✅ Entity linking listo (grafo={ENTITY_SOURCE}, ENTITY_MIN_COS={ENTITY_MIN_COS})")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[  chunks] 7 entidades indexadas


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

[  hybrid] 339 entidades indexadas


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

[original] 508 entidades indexadas

✅ Entity linking listo (grafo=original, ENTITY_MIN_COS=0.85)


In [28]:
# ─── Router + Entity linking ─────────────────────────────────────────

def retrieve_router_entity(query, top_n=TOP_N):
    """Router M1 + entity linking M3 en transversales."""
    label = classify_query(query)
    if label == "transversal":
        return retrieve_graph_rag_entity(query, _entity_variant, top_n=top_n)
    return retrieve_baseline(query, top_n=top_n)


def retrieve_router_entity_k(query, k):
    return retrieve_router_entity(query, top_n=k)


def generate_entity_answer(query, top_n=TOP_N, use_router=False):
    """Generación con entity linking (boost_noprompt: sin triples en prompt)."""
    if use_router:
        label = classify_query(query)
        if label == "factual":
            chunks = retrieve_baseline(query, top_n=top_n)
            return generate_answer(build_prompt_baseline(query, chunks)), chunks, [], label
        chunks = retrieve_graph_rag_entity(query, _entity_variant, top_n=top_n)
        g, dbg = retrieve_graph_entity(query, _entity_variant, k=GRAPH_TOP_K)
        prompt = build_prompt_baseline(query, chunks)
        return generate_answer(prompt), chunks, g, label
    chunks = retrieve_graph_rag_entity(query, _entity_variant, top_n=top_n)
    g, dbg = retrieve_graph_entity(query, _entity_variant, k=GRAPH_TOP_K)
    prompt = build_prompt_baseline(query, chunks)
    return generate_answer(prompt), chunks, g, "entity"


# ─── Sanity check: megafonía (caso patológico) ───────────────────────
_mega_q = "Estado de las instalaciones de megafonía"
print(f"🔎 Entity linking debug: {_mega_q!r}\n")

g_cos = retrieve_graph_variant(_mega_q, _entity_variant, k=GRAPH_TOP_K)
g_ent, dbg = retrieve_graph_entity(_mega_q, _entity_variant, k=GRAPH_TOP_K)

print(f"  Entidades query:     {dbg['query_entities']}")
print(f"  Entidades grafo:     {dbg['matched_graph_entities']}")
print(f"\n  Triples COSENO (top-{GRAPH_TOP_K}):")
for t in g_cos:
    flag = "✓boost" if t["score"] >= GRAPH_MIN_COS else " skip "
    print(f"    [{flag}] cos={t['score']:.3f} | {t['text'][:80]}")
print(f"\n  Triples ENTITY-LINK ({len(g_ent)} tras filtro):")
if not g_ent:
    print("    (ninguno → grafo DESACTIVADO para esta query, boost = baseline)")
else:
    for t in g_ent:
        print(f"    cos={t['score']:.3f} | {t['text'][:80]}")


🔎 Entity linking debug: 'Estado de las instalaciones de megafonía'

  Entidades query:     []
  Entidades grafo:     []

  Triples COSENO (top-5):
    [✓boost] cos=0.818 | estructura metálica tiene desoxidado
    [✓boost] cos=0.818 | trabajos de instalación involucra protección de grava
    [✓boost] cos=0.818 | constructora solicita ficha técnica de los paneles de aislamiento
    [✓boost] cos=0.816 | Dirección de obra informa que se han de desmontar los bajantes existentes
    [✓boost] cos=0.816 | Trabajos de limpieza de la estructura metálica de marquesinas

  Triples ENTITY-LINK (0 tras filtro):
    (ninguno → grafo DESACTIVADO para esta query, boost = baseline)


In [29]:
# ─── Eval retrieval M3 ───────────────────────────────────────────────
print("📏 Evaluando retrieval · graph_entity_boost_noprompt...")
ent_summary, ent_detail = eval_variant("graph_entity_boost_noprompt", retrieve_graph_entity_k, BENCHMARK_QUERIES)

print("📏 Evaluando retrieval · graph_router_entity...")
router_ent_summary, router_ent_detail = eval_variant("graph_router_entity", retrieve_router_entity_k, BENCHMARK_QUERIES)

# Comparativa con referencias clave
_refs = []
try:
    _refs = [s for s in all_summaries if s["variant"] in ("baseline", "graph_router")]
except NameError:
    print("⚠️  Ejecuta sec 8 y 11 antes para tener baseline/router de referencia")

df_m3_ret = pd.DataFrame(_refs + [ent_summary, router_ent_summary]).set_index("variant")
print("\n🏆 Retrieval M3 vs referencias:")
display(df_m3_ret.sort_values("NDCG@5", ascending=False))

# Subsets factual / transversal
_m3_details = {"graph_entity_boost_noprompt": ent_detail, "graph_router_entity": router_ent_detail}
try:
    _m3_details["baseline"] = all_details["baseline"]
    _m3_details["graph_router"] = router_ret_detail
except NameError:
    pass

print("\n🎯 FACTUAL:")
display(pd.DataFrame([
    _summary_subset(n, d, BENCHMARK_QUERIES_FACTUAL)
    for n, d in _m3_details.items()
]).set_index("variant").sort_values("NDCG@5", ascending=False))

print("\n🌐 TRANSVERSAL:")
display(pd.DataFrame([
    _summary_subset(n, d, BENCHMARK_QUERIES_TRANSVERSAL)
    for n, d in _m3_details.items()
]).set_index("variant").sort_values("NDCG@5", ascending=False))

# Δ NDCG@5: entity vs coseno puro (mismo grafo)
try:
    _cos_name = f"graph_{ENTITY_SOURCE}_boost"
    print(f"\nΔ NDCG@5 por query (entity - {_cos_name}):")
    _delta = [{
        "query": q[:55],
        "entity": ent_detail[i]["ndcg@5"],
        "coseno": all_details[_cos_name][i]["ndcg@5"],
        "Δ": ent_detail[i]["ndcg@5"] - all_details[_cos_name][i]["ndcg@5"],
    } for i, q in enumerate(BENCHMARK_QUERIES)]
    display(pd.DataFrame(_delta).sort_values("Δ", ascending=False))
except (NameError, KeyError):
    print("(Ejecuta sec 8 para comparar entity vs coseno)")


📏 Evaluando retrieval · graph_entity_boost_noprompt...
📏 Evaluando retrieval · graph_router_entity...

🏆 Retrieval M3 vs referencias:


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
graph_router_entity,0.771429,0.905353,0.638095,0.898198,0.900794,0.853855
graph_entity_boost_noprompt,0.761905,0.904798,0.633333,0.895641,0.900794,0.849292
baseline,0.752381,0.899905,0.628571,0.891062,0.900794,0.842912



🎯 FACTUAL:


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
graph_router,0.916667,0.990980,0.716667,0.978785,1.0,0.979812
graph_entity_boost_noprompt,0.916667,0.982344,0.716667,0.969133,1.0,0.979812
graph_router_entity,0.916667,0.979663,0.716667,0.966452,1.0,0.979812
baseline,0.916667,0.979663,0.716667,0.967469,1.0,0.979812



🌐 TRANSVERSAL:


,Precision@5,NDCG@5,Precision@10,NDCG@10,MRR,MAP
variant,,,,,,
graph_router,0.600000,0.828806,0.533333,0.788736,0.740741,0.691434
graph_router_entity,0.577778,0.806273,0.533333,0.807193,0.768519,0.685913
graph_entity_boost_noprompt,0.555556,0.801404,0.522222,0.797651,0.768519,0.675265
baseline,0.533333,0.793561,0.511111,0.789185,0.768519,0.660378



Δ NDCG@5 por query (entity - graph_original_boost):


,query,entity,coseno,Δ
3,¿Qué responsable está asignado a las acciones ...,1.000000,0.000000,1.000000
10,¿Qué certificados de material debe aportar la ...,0.985227,0.585982,0.399245
6,¿Qué documentación debe aportar la UTE sobre e...,1.000000,0.712263,0.287737
5,Estado de las instalaciones de megafonía,0.896375,0.734493,0.161882
15,Agrupa todas las incidencias detectadas en las...,0.919721,0.760640,0.159081
2,¿Qué incidencias AR-29 aparecen?,1.000000,0.867087,0.132913
16,¿Cuáles son las acciones pendientes más frecue...,0.852928,0.732829,0.120099
7,¿Cuál es la función de la sigla DEO en un acta...,1.000000,0.888722,0.111278
11,¿A qué altura respecto al suelo se van a insta...,1.000000,0.900313,0.099687
13,¿Qué elementos constructivos presentan más inc...,1.000000,0.904717,0.095283


In [30]:
# ─── Eval calidad de RESPUESTA M3 ────────────────────────────────────
if ANSWER_EVAL_ENABLED:
    m3_ans_rows = {}

    print("📏 RESPUESTAS · graph_entity_boost_noprompt...")
    rows = []
    for q in BENCHMARK_QUERIES:
        ans, chunks, triples, _ = generate_entity_answer(q, use_router=False)
        ctx = [strip_doc_prefix(c["text"], EMBED_BACKEND) for c in chunks]
        rows.append({"query": q[:60], **judge_answer(q, ans, ctx)})
    m3_ans_rows["graph_entity_boost_noprompt"] = rows

    print("📏 RESPUESTAS · graph_router_entity...")
    rows = []
    for q in BENCHMARK_QUERIES:
        ans, chunks, triples, label = generate_entity_answer(q, use_router=True)
        ctx = [strip_doc_prefix(c["text"], EMBED_BACKEND) for c in chunks]
        rows.append({"query": q[:60], "label": label, **judge_answer(q, ans, ctx)})
    m3_ans_rows["graph_router_entity"] = rows

    m3_summaries = [summarize_answer_rows(n, r) for n, r in m3_ans_rows.items()]

    try:
        df_m3_ans = pd.DataFrame(answer_summaries + m3_summaries).set_index("variant")
    except NameError:
        df_m3_ans = pd.DataFrame(m3_summaries).set_index("variant")

    print("\n🏆 Calidad de RESPUESTA — M3 vs resto:")
    display(df_m3_ans.sort_values("overall", ascending=False))

    print("\n🌐 TRANSVERSAL:")
    _t_rows = []
    try:
        for name, all_rows in answer_details.items():
            sub = [r for r, q in zip(all_rows, BENCHMARK_QUERIES) if q in BENCHMARK_QUERIES_TRANSVERSAL]
            _t_rows.append(summarize_answer_rows(name, sub))
    except NameError:
        pass
    for name, rows in m3_ans_rows.items():
        sub = [r for r, q in zip(rows, BENCHMARK_QUERIES) if q in BENCHMARK_QUERIES_TRANSVERSAL]
        _t_rows.append(summarize_answer_rows(name, sub))
    display(pd.DataFrame(_t_rows).set_index("variant").sort_values("overall", ascending=False))
else:
    print("ANSWER_EVAL_ENABLED=False → saltado")


📏 RESPUESTAS · graph_entity_boost_noprompt...
📏 RESPUESTAS · graph_router_entity...

🏆 Calidad de RESPUESTA — M3 vs resto:


,correctness,faithfulness,citation,overall
variant,,,,
graph_entity_boost_noprompt,1.761905,1.952381,1.333333,1.764286
baseline,1.714286,1.904762,1.476190,1.745238
graph_router_entity,1.714286,1.904762,1.333333,1.723810
graph_chunks_boost,1.761905,1.952381,1.047619,1.721429
graph_chunks_ctxonly,1.714286,1.952381,1.000000,1.690476
graph_chunks_boost_noprompt,1.666667,1.904762,1.142857,1.671429
graph_hybrid_boost_noprompt,1.619048,1.952381,1.047619,1.650000
graph_original_boost_noprompt,1.619048,1.904762,1.142857,1.647619
graph_hybrid_ctxonly,1.571429,1.809524,1.142857,1.590476



🌐 TRANSVERSAL:


,correctness,faithfulness,citation,overall
variant,,,,
graph_chunks_boost,1.666667,2.000000,0.888889,1.666667
graph_entity_boost_noprompt,1.444444,1.888889,1.222222,1.566667
graph_chunks_ctxonly,1.444444,1.888889,0.888889,1.516667
graph_chunks_boost_noprompt,1.444444,1.888889,0.888889,1.516667
baseline,1.333333,1.777778,1.222222,1.472222
graph_hybrid_boost_noprompt,1.333333,1.888889,0.888889,1.461111
graph_router_entity,1.333333,1.777778,1.111111,1.455556
graph_original_boost_noprompt,1.222222,1.777778,1.000000,1.383333
graph_hybrid_boost,1.222222,1.666667,0.888889,1.327778
